# 00. Agent Trace → 평가 데이터셋 준비 (Dataset Preparation)

## 📌 이 노트북의 목표

대부분의 조직은 **AI Agent를 운영 중**이고 **trace 로그도 쌓이고 있지만**,  
정작 **그 로그를 가지고 평가를 어떻게 시작해야 할지 막막**합니다.

특히 가장 큰 두 가지 장벽은:

1. **샘플링** — 운영 trace 전부를 평가하기엔 비싸고, 무작위로 뽑으면 편향됨. 무엇을 얼마나 뽑아야 통계적으로 의미 있나?
2. **레이블링** — 정답이 없는 운영 로그에서 ground truth를 어떻게 만드나? 사람 검수가 필요한 곳은 어디인가?

이 핸즈온은 그 간극을 메웁니다.

```
[고객이 보유한 것]                    [이 노트북이 만드는 것]
운영 중 AI Agent  ──→  trace log  ──→  RAG 평가 데이터셋   ──→  01~05, 07 노트북
                                  └──→  Agent 평가 데이터셋  ──→  06, 08 노트북
```

이 노트북에서 만든 데이터셋이 그대로 **01_QA_Evaluator ~ 08_Agent_Evaluation** 의 입력이 됩니다.

## 🎯 도메인 시나리오

본 핸즈온은 **은행 고객 상담 Agent** 시나리오를 사용합니다.

| 도메인 | 하위 도메인 |
|------|--------|
| 은행 고객 상담 Agent | savings · loan · card · account |

> 본 튜토리얼의 모든 노트북(00~08)은 은행 도메인 데이터로 일관되게 동작합니다.  
> 자기 도메인 trace를 가져오시려면 3단계의 BYO trace 옵션을 사용하세요.

## 🪜 워크플로우

```
[3]  Agent Trace 확보  (BYO trace 로드  또는  시뮬레이션)
        ↓ [4]  PII 제거 / 비식별화
        ↓ [5]  탐색적 데이터 분석 (EDA)
        ↓ [6]  데이터 정제 (Cleaning)
        ↓ [7]  대화 → 턴 단위 변환
        ↓ [7.5] ⭐ 통계적 샘플링 (표본크기 산정·다차원 층화·다양성·실패 oversample)
        ↓ [8]  평가 스키마 변환
        ↓ [8.5] ⭐ Ground Truth 레이블링 (Rubric · LLM-assisted · 검수 큐 · IAA · Agent expected_tool_calls)
        ↓ [9]  데이터 분할 (Dev / Test / Holdout)
        ↓ [10] RAG 평가용 내보내기  →  data/sample_qa.json, data/eval_input.jsonl …
        ↓ [11] Agent Trace 형식 변환 →  data/agent_eval_traces.jsonl
        ↓ [12] 증분 업데이트 시나리오
```


## 1단계: 필수 패키지 설치

In [1]:
%pip install pandas numpy scikit-learn python-dotenv --quiet


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2단계: 라이브러리 임포트

In [2]:
import os
import json
import random
import hashlib
import re
from datetime import datetime, timedelta
from collections import Counter
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 재현성을 위한 시드 고정
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

print('✓ 라이브러리 로드 완료')
print('✓ 도메인: 은행 고객 상담 Agent (savings · loan · card · account)')


✓ 라이브러리 로드 완료
✓ 도메인: 은행 고객 상담 Agent (savings · loan · card · account)


## 3단계: Agent Trace 확보

이 단계는 두 가지 진입로를 지원합니다.

### 🅰️ 옵션 A — 이미 운영 trace 로그가 있는 경우 (실무 기본 경로)

조직이 보유한 trace를 `data/raw_chat_logs.jsonl` 위치에 두고, 환경변수 `USE_EXISTING_LOG=true`로 실행하면 시뮬레이션을 건너뜁니다.

**최소 요구 컬럼 매핑** — 보유 로그를 아래 표준 스키마로 평탄화해 주세요. 1행 = 1턴.

| 표준 필드 | 타입 | 의미 | 보유 trace에서 끌어올 위치(예) |
|---|---|---|---|
| `session_id` | string | 대화 세션 ID | OTel `trace_id` / Foundry `thread_id` |
| `turn_index` | int | 세션 내 턴 번호(0부터) | message 순서 |
| `timestamp` | string (ISO 8601) | 턴 시각 | span start time |
| `user_id` | string | (선택) 사용자 식별자 (해시 권장) | request header / claims |
| `user_message` | string | 사용자 발화 | role=user content |
| `assistant_message` | string | 최종 어시스턴트 텍스트 | 마지막 role=assistant content |
| `retrieved_context` | string | RAG 검색 결과 / 도구가 반환한 근거 텍스트 | tool 결과 또는 retriever output |
| `tool_calls` | list[dict] | tool_name / tool_input / tool_output / status / duration_ms | function-calling spans |
| `model_name` | string | 사용된 모델 | model deployment id |
| `latency_ms` | int | (선택) | span duration |
| `token_usage` | dict | (선택) input/output/total | usage 필드 |
| `feedback_score` | int 1~5 | (선택, 부분만) 사용자 피드백 | thumbs up/down 변환 |
| `error_flag` | bool | (선택) 응답 오류 여부 | exception / non-2xx |

> ⚠️ Agent 평가는 **tool_calls 유무**로 가능한 평가가 크게 갈립니다. RAG QA만 평가할 거면 `tool_calls`는 비워도 됩니다.

### 🅱️ 옵션 B — trace가 없거나 학습용으로 모사하고 싶은 경우

본 노트북의 시뮬레이터가 위 스키마와 동일한 `raw_chat_logs.jsonl`을 생성합니다. 옵션 A로 자기 데이터에 그대로 적용할 수 있습니다.

---

### 실제 Agent Trace의 구성 요소 (참고)

```
Session (trace_id)
 ├─ metadata: agent_name, agent_version, model, latency, tokens
 ├─ messages[]
 │   ├─ system       (role instruction)
 │   ├─ user         (사용자 질문)
 │   ├─ assistant    (reasoning + tool_calls)
 │   ├─ tool         (tool 실행 결과 - 대부분 markdown)
 │   ├─ assistant    (reasoning + tool_calls)   ← 필요시 반복
 │   ├─ tool         (tool 실행 결과)
 │   └─ assistant    (최종 응답)
 └─ feedback (있을 수도 / 없을 수도)
```


In [3]:
# ============================================================
# 은행 고객 상담 Agent 데이터 풀
#   - QA_POOL                 : 도메인별 질문/컨텍스트/정답 풀
#   - DOMAIN_SYSTEM_PROMPTS   : 도메인별 system prompt
#   - TOOL_CALL_TEMPLATES     : 도메인별 도구 호출 템플릿
#   - DUMMY_*                 : PII 시뮬레이션용 더미 값
#   - HALLUCINATION_TEMPLATES : 환각 응답 시뮬레이션용
#   - WRONG_CONTEXTS          : 잘못된 RAG 결과 시뮬레이션용
# ============================================================

QA_POOL = [
    {
        'domain': 'savings',
        'questions': [
            '적금 금리가 어떻게 되나요?', '정기예금 만기 후 어떻게 되나요?',
            '적금 중도해지하면 이자는?', '자유적금이랑 정기적금 차이가 뭐야?',
            '예금자보호 한도가 얼마야?', '복리 적금 상품 있어?',
            '적금 금리 비교 좀 해줘', '청년 우대 적금 가입 조건은?',
            '예적금 만기 후 자동연장은 어떻게 돼?', '비과세 예금 가입 조건은?',
        ],
        'contexts': [
            '[(SAVINGS_RATE_TABLE_2025)]\n- 정기적금 12개월: 3.5% p.a. (우대 +0.3%, 최대 3.8%)\n- 정기적금 24개월: 3.8% p.a. (우대 +0.3%, 최대 4.1%)\n- 정기적금 36개월: 4.0% p.a.\n- 자유적금: 2.8% p.a. (자동이체 시 +0.2%)\n- 청년 우대 적금: 5.0% p.a. (만 19~34세, 한도 50만원/월)\n- 우대조건: 급여이체, 카드 사용, 자동이체 등록\n- 적용기간: 2025-01-01 ~ 2025-06-30\n- 최소 가입금액: 1만원/월',
            '[(DEPOSIT_AUTO_RENEWAL_POLICY)]\n- 만기 후 별도 해지 없으면 자동 연장\n- 연장 시점 금리 적용 (가입 시점 금리 아님)\n- 연장 기간: 기존 약정 기간과 동일\n- 만기 14일 전 SMS/앱 알림 발송\n- 자동연장 해제: 앱 메뉴 > 적금 > 만기관리\n- 영업점 방문 시 즉시 해제 가능\n- 연장된 금리는 우대금리 미반영 (기본금리만 적용)\n- 연장 후 중도해지 시 연장 시점부터 기간 계산',
            '[(SAVINGS_EARLY_TERMINATION_POLICY)]\n- 1개월 미만: 0.1% p.a.\n- 1~6개월: 약정금리 50%\n- 6~12개월: 약정금리 70%\n- 12개월 이상: 약정금리 80%\n- 우대금리는 모두 미적용\n- 비과세 적용분도 일반과세로 환산\n- 신청 방법: 앱 또는 영업점\n- 처리 시간: 신청 즉시 (영업시간 내)\n- 입금된 원금은 연결계좌로 자동 송금',
            '[(SAVINGS_PRODUCT_COMPARISON)]\n- 자유적금: 납입 자유, 금리 낮음 (연 2.8%)\n  · 권장 대상: 비정기 소득자, 단기 목적자금\n  · 한도: 1회 최대 100만원, 월 300만원\n- 정기적금: 매월 정액, 금리 높음 (연 3.5~4.0%)\n  · 권장 대상: 급여 소득자, 목돈 마련\n  · 자동이체 등록 시 우대 +0.2%\n- 정기예금: 목돈 예치, 안정적 (연 3.2~3.8%)\n  · 권장 대상: 여유자금 운용\n  · 최소 100만원 이상\n- 모든 상품: 비과세 한도 5,000만원 적용 가능 (조건부)',
            '[(DEPOSIT_INSURANCE_FAQ)]\n- 예금자보호 한도: 1인당 5천만원\n- 이자 포함 5천만원 (원금 + 약정이자)\n- 보호 대상: 예금, 적금, 신탁, CD, 보장성 보험\n- 비보호 상품: 펀드, ELS, 외화예금(일부), 후순위채권\n- 동일 은행 내 합산 (지점 무관)\n- 은행별 별도 적용 (A은행 5천만 + B은행 5천만 = 1억 보호)\n- 예금보험공사 신청 절차: 부보 사유 발생 후 안내\n- 지급 시기: 통상 7영업일 내\n- 5천만원 초과분은 파산절차에 따라 회수 (일부 가능)',
        ],
        'good_responses': [
            '현재 정기적금 금리는 12개월 기준 연 3.5%, 24개월 기준 연 3.8%입니다. 자유적금은 연 2.8%입니다.',
            '만기 후 별도 해지 요청이 없으면 동일 기간으로 자동 연장되며, 연장 시점의 금리가 적용됩니다.',
            '중도해지 시 약정금리가 아닌 중도해지 이율이 적용됩니다. 1개월 미만은 0.1%, 1~6개월은 약정금리의 50%가 적용됩니다.',
            '자유적금은 매월 납입금액을 자유롭게 설정할 수 있지만 금리가 낮고, 정기적금은 매월 정해진 금액을 납입하지만 금리가 높습니다.',
            '예금자보호 한도는 1인당 원금과 이자를 합산하여 최대 5천만원까지입니다.',
        ],
    },
    {
        'domain': 'loan',
        'questions': [
            '주택담보대출 금리 알려줘', '신용대출 한도 조회하면 신용점수에 영향 있어?',
            '전세자금대출 조건이 뭐야?', '대출 중도상환수수료는?',
            '대환대출이 뭐야?', 'DSR 규제가 뭔가요?',
            '신용대출 금리 비교해줘', '대출 연장 절차는 어떻게 돼?',
            '마이너스 통장 개설 조건은?', '학자금 대출 상환 방법은?',
        ],
        'contexts': [
            '[(MORTGAGE_RATE_TABLE_2025)]\n- 변동금리: 3.8~5.2% p.a. (COFIX 연동, 6개월 변동)\n- 고정금리(5년): 4.5~5.5% p.a.\n- 고정금리(10년): 4.8~5.8% p.a.\n- 혼합금리: 4.2~5.0% p.a. (5년 고정 후 변동)\n- 우대금리: 급여이체 -0.1%, 카드사용 -0.05%, 자동이체 -0.05%\n- 최대 우대: -0.5% (소득 증빙 시 추가)\n- LTV 한도: 무주택 70%, 1주택 60%, 다주택 40%\n- DTI 한도: 40%, DSR 40% (총부채원리금상환비율)\n- 적용 기간: 2025-01-01 ~ 2025-03-31',
            '[(CREDIT_INQUIRY_IMPACT_FAQ)]\n- Soft inquiry (한도 조회): 신용점수 영향 없음\n- Hard inquiry (정식 신청): 1~3점 일시적 하락 가능\n- 14일 내 동일 상품 복수 조회는 1건으로 처리 (Rate shopping)\n- 6개월 내 5회 이상 조회 시 추가 감점 가능\n- 조회 영향 회복 기간: 통상 3~6개월\n- 본인 조회는 신용점수 영향 없음 (월 1회 무료)\n- 카드 발급 신청도 Hard inquiry에 포함\n- 대출 거절 사유 중 "조회 과다"는 해당 항목\n- 최근 1년 조회 내역은 신용평가에 반영',
            '[(JEONSE_LOAN_ELIGIBILITY_2025)]\n- 전세계약서 + 확정일자 필요 (필수)\n- 소득 증빙 (급여명세서/소득금액증명)\n- 대출금액: 보증금의 최대 80%\n- 한도: 수도권 5억, 지방 3억\n- 금리: 연 3.5~4.5% (HUG/HF 보증 기준)\n- 보증기관: HUG(주택도시보증공사) 또는 HF(한국주택금융공사)\n- 보증료: 연 0.05~0.1% (HUG), 연 0.06~0.1% (HF)\n- 청년 전세대출: 만 34세 이하, 우대금리 -0.3%\n- 신혼부부: 결혼 7년 이내, 우대금리 -0.2%\n- 대출 기간: 전세 계약 기간 (최대 2년, 갱신 가능)',
            '[(MORTGAGE_PREPAYMENT_FEE_RULES)]\n- 수수료율: 1.2~1.5% (잔존 원금 기준)\n- 거치기간: 보통 3년\n- 거치기간 후: 수수료 면제\n- 일부상환 가능 (해당 금액만 수수료)\n- 면제 케이스: 사망/장애/실직 (서류 제출 필요)\n- 계산 예시: 1억 잔존 → 3년 내 상환 시 120만원 수수료\n- 매년 잔존원금의 10%까지는 수수료 없이 일부상환 가능\n- 신청: 앱 또는 영업점, 처리 시간 1~2영업일\n- 대환 목적 상환: 일부 은행 수수료 우대\n- 변동금리 → 고정금리 전환 시 수수료 면제 가능',
            '[(LOAN_REFINANCING_GUIDE)]\n- 대환대출: 기존 대출을 새 대출로 갈아타기\n- 조건: 6개월 이상 정상 상환\n- 장점: 금리 인하, 조건 개선\n- 한도: 기존 대출 잔액 + 추가한도(소득에 따라 다름)\n- 절차: 1) 한도조회 → 2) 신청 → 3) 심사 → 4) 기존 대출 상환 → 5) 신규 대출 실행\n- 처리 기간: 통상 5~10영업일\n- 주의사항: 중도상환수수료 발생 가능 (거치기간 내)\n- 대환 우대 상품: 인터넷 전용 -0.2%\n- 정부 지원 대환대출: 햇살론, 안심전환대출 등\n- 신용점수 영향: 신규 신청으로 일시 하락 가능 (1~3점)',
        ],
        'good_responses': [
            '주택담보대출 금리는 변동금리 기준 연 3.8~5.2%, 고정금리(5년) 기준 연 4.5~5.5%입니다.',
            '단순 한도 조회(Soft inquiry)는 신용점수에 영향을 주지 않습니다. 실제 대출 신청(Hard inquiry) 시에만 1~3점 일시적으로 하락할 수 있습니다.',
            '전세자금대출은 전세계약서와 확정일자가 필요하며, 소득 증빙 후 보증금의 최대 80%까지 대출 가능합니다.',
            '중도상환수수료는 상환금액의 1.2~1.5%이며, 대출 후 3년(거치기간)이 지나면 면제됩니다.',
            '대환대출은 기존 대출을 금리가 낮은 새 대출로 갈아타는 것입니다. 6개월 이상 정상 상환 이력이 필요합니다.',
        ],
    },
    {
        'domain': 'card',
        'questions': [
            '체크카드 해외결제 수수료 있어?', '신용카드 포인트 사용법은?',
            '카드 분실 신고 어떻게 해?', '카드 한도 변경은 어떻게?',
            '연회비 면제 조건은?', '카드 결제일 변경 가능해?',
            '해외에서 카드 사용 시 환율은?', '카드 재발급 절차는?',
        ],
        'contexts': [
            '[(DEBIT_CARD_FX_FEE_SCHEDULE)]\n- 국제브랜드 수수료: ~1.0% (Visa/Master 기준)\n- 해외서비스 수수료: 0.2~0.5% (은행별)\n- 환율: 결제일 기준 적용 (전일 종가 + 마진)\n- 통화별 적용: USD/EUR/JPY 등 주요 통화 동일\n- 일부 통화(태국 바트, 베트남 동): 추가 0.3% 수수료\n- 해외 ATM 출금 수수료: 건당 미화 3달러 + 인출 금액의 1%\n- 일일 한도: 미화 5,000달러 상당\n- 사전 알림: 해외 사용 전 카드사 앱에서 등록 권장\n- 분실/도난 시 24시간 글로벌 콜센터 이용\n- 무 수수료 환전: 일부 프리미엄 카드 한정 혜택',
            '[(CREDIT_CARD_POINT_GUIDE)]\n- 포인트 사용: 결제 차감, 현금 전환, 기프트카드\n- 최소 사용: 1,000포인트\n- 유효기간: 적립일로부터 5년\n- 적립률: 일반 가맹점 0.5~1%, 특별 가맹점 2~5%\n- 카드별 차이: 일반카드 0.5%, 골드 1%, 플래티넘 1.5%\n- 현금 전환: 1포인트 = 1원, 환전 수수료 없음\n- 가족 합산 사용: 본인 + 배우자 + 자녀 포인트 통합 가능\n- 만료 임박 알림: 만료 30일 전 앱 푸시\n- 포인트 양도: 동일 카드사 내에서만 가능\n- 세금 신고: 50만 포인트 이상 사용 시 기타소득 처리',
            '[(CARD_LOSS_REPORT_PROC)]\n- 앱: 카드 > 분실신고\n- 전화: 1588-XXXX (24시간)\n- 신고 즉시 사용 정지\n- 부정사용 보상 신청 가능 (분실 60일 내)\n- 재발급 신청: 분실신고와 동시에 가능\n- 재발급 비용: 일반 2,000원, 골드 이상 면제\n- 배송 기간: 영업일 기준 3~5일\n- 임시 카드: 영업점 방문 시 즉시 발급 가능 (1주일 한도 50만)\n- 부정사용 조사 기간: 통상 30일\n- 보상 한도: 분실신고 시점 이전 60일 사용분 (본인 과실 제외)\n- 비밀번호 노출 시 보상 불가',
            '[(CARD_LIMIT_CHANGE_GUIDE)]\n- 앱: 카드 > 이용한도 > 변경 신청\n- 한도 상향: 소득증빙 필요 (재직증명서/소득금액증명)\n- 한도 하향: 즉시 적용\n- 일시적 한도 상향: 최대 3개월 (해외여행/이사 등)\n- 한도 산정 기준: 연소득의 30~50% (신용등급에 따라)\n- 처리 기간: 상향 1~3영업일, 하향 즉시\n- 6개월 이내 재상향 신청 제한 (일시 상향은 별도)\n- 카드론 한도와 별도 운영\n- 해외 한도: 국내 한도 내에서 별도 설정 가능\n- 가족카드 한도는 본인 카드 한도 내 설정',
        ],
        'good_responses': [
            '체크카드 해외결제 시 국제브랜드 수수료 약 1%와 해외서비스 수수료 0.2~0.5%가 부과됩니다.',
            '신용카드 포인트는 결제 차감, 현금 전환, 기프트카드 교환 등으로 사용할 수 있습니다. 최소 1,000포인트부터 사용 가능합니다.',
            '카드 분실 시 모바일앱의 카드 메뉴에서 분실신고하거나, 고객센터(1588-XXXX)로 24시간 신고 가능합니다. 신고 즉시 사용이 정지됩니다.',
            '카드 한도 변경은 모바일앱에서 신청할 수 있습니다. 한도 상향은 소득증빙이 필요하며, 하향은 즉시 적용됩니다.',
        ],
    },
    {
        'domain': 'account',
        'questions': [
            '계좌 개설은 어떻게 해?', 'ATM 출금 한도가 얼마야?',
            '자동이체 해지 방법은?', '계좌가 압류되면 어떻게 해?',
            '모바일뱅킹 해외에서 사용 가능해?', '공동인증서 갱신은 어떻게?',
            '타행 이체 수수료는?', '통장 비밀번호 변경은?',
        ],
        'contexts': [
            '[(ACCOUNT_OPENING_GUIDE)]\n- 비대면: 앱 다운로드 > 본인인증 > 신분증 촬영\n- 대면: 영업점 방문, 신분증 지참\n- 필요서류: 신분증, 기본증명서(미성년)\n- 영업시간: 평일 09:00~16:00 (영업점 기준)\n- 비대면 가능 시간: 24시간 (점검 시간 02:00~04:00 제외)\n- 계좌 종류: 입출금자유, 정기예금, 정기적금, 외화예금, CMA\n- 처리 시간: 비대면 10~15분, 대면 30분 내외\n- 1인당 동일은행 비대면 계좌 한도: 입출금 3개\n- 미성년자: 법정대리인 동행 또는 동의서 필요\n- 외국인: 외국인등록증 + 체류 자격 확인\n- 개설 후 즉시 인터넷뱅킹/체크카드 신청 가능',
            '[(ATM_WITHDRAWAL_LIMITS_2025)]\n- 1회: 100만원\n- 1일: 300만원\n- 한도 상향: 앱/영업점 신청 (최대 600만원/일)\n- 타행 ATM 수수료: 900~1,000원\n- 영업시간 외 (18:00~익일 08:30): +500원\n- 토요일/공휴일: 종일 야간 수수료 적용\n- 보안카드/OTP 사용 시 한도 상향 가능\n- 자행 ATM: 24시간 무료\n- 출금 한도 변경 처리: 신청 즉시 적용\n- 소액 출금 (1만원 미만): 일부 ATM 미지원\n- 5만원권 출금: ATM 잔량 따라 제한 가능\n- 해외 ATM: 별도 한도, 미화 5,000달러 상당',
            '[(AUTO_TRANSFER_MGMT_GUIDE)]\n- 해지: 앱 > 이체 > 자동이체 > 해지\n- 변경: 출금일 전일 23:59까지\n- 다음 실행일부터 적용\n- 등록 절차: 앱 > 이체 > 자동이체 > 신규 등록\n- 등록 시 필요 정보: 받는 분 계좌, 금액, 주기, 시작일\n- 잔액 부족 시: 당일 미이체, 다음 영업일 재시도 (최대 3회)\n- 3회 재시도 실패: 자동이체 일시 정지, SMS 알림\n- 등록 한도: 1계좌당 최대 50건\n- 수수료: 자행 무료, 타행 500원 (영업시간 외 1,000원)\n- 공과금/카드 자동이체는 우대 (수수료 면제)\n- 해지 후 재등록 가능, 이력 30일 보관',
            '[(ACCOUNT_SEIZURE_FAQ)]\n- 압류 금액 초과분 출금 가능\n- 생계비(월 185만원) 보호 신청 가능\n- 법원 압류범위변경 신청 필요\n- 압류 통지: SMS 및 등기우편으로 통보\n- 압류 사유 확인: 법원/세무서 등 압류기관 문의\n- 해제 절차: 채무 변제 → 압류기관 해제 신청 → 은행 통보\n- 해제 처리 기간: 통상 3~5영업일\n- 임금/연금 압류: 1/2 보호 (민사집행법)\n- 보장성 보험 환급금: 일부 압류 제한\n- 미성년자 명의 계좌: 친권자 동의 시에만 압류 가능\n- 압류 중 신규 입금: 즉시 압류 대상 (생계비 제외)\n- 변호사/법무사 상담 권장',
        ],
        'good_responses': [
            '비대면 계좌 개설은 모바일앱에서 본인인증과 신분증 촬영을 통해 가능합니다. 영업점 방문 시에는 신분증을 지참하시면 됩니다.',
            'ATM 1회 출금 한도는 100만원, 1일 한도는 300만원입니다. 앱이나 영업점에서 최대 600만원/일까지 상향 신청이 가능합니다.',
            '자동이체 해지는 모바일앱의 이체 > 자동이체 메뉴에서 가능합니다. 출금 예정일 전일 23:59까지 처리해야 합니다.',
            '압류된 계좌는 압류 금액 초과분에 대해 출금이 가능합니다. 생계비(월 185만원)는 법원에 압류범위변경 신청을 통해 보호받을 수 있습니다.',
        ],
    },
]

DOMAIN_SYSTEM_PROMPTS = {
    'savings': '당신은 은행 적금/예금 상담 에이전트입니다. 고객의 질문에 정확하게 답변하세요. 반드시 제공된 도구를 사용하여 정보를 조회하세요.',
    'loan':    '당신은 은행 대출 상담 에이전트입니다. 고객의 금융 질문에 정확하게 답변하세요. 반드시 제공된 도구를 사용하여 정보를 조회하세요.',
    'card':    '당신은 은행 카드 상담 에이전트입니다. 고객의 카드 관련 질문에 정확하게 답변하세요. 반드시 제공된 도구를 사용하세요.',
    'account': '당신은 은행 계좌 상담 에이전트입니다. 고객의 계좌 관련 질문에 정확하게 답변하세요. 반드시 제공된 도구를 사용하세요.',
}

TOOL_CALL_TEMPLATES = {
    'savings': [
        {'tool_name': 'search_knowledge_base', 'tool_input': {'query': '적금 금리 정보', 'top_k': 3}, 'tool_output': '검색 결과 3건 반환'},
        {'tool_name': 'get_product_detail', 'tool_input': {'product_id': 'SAV-2025-001'}, 'tool_output': '정기적금 12개월 상품 정보 반환'},
        {'tool_name': 'calculate_interest', 'tool_input': {'principal': 1000000, 'rate': 3.5, 'months': 12}, 'tool_output': '예상 이자: 35,000원'},
    ],
    'loan': [
        {'tool_name': 'search_knowledge_base', 'tool_input': {'query': '대출 금리 조건', 'top_k': 3}, 'tool_output': '검색 결과 3건 반환'},
        {'tool_name': 'check_eligibility', 'tool_input': {'product_type': 'mortgage', 'income': 50000000}, 'tool_output': '대출 가능 여부: True'},
        {'tool_name': 'get_rate_table', 'tool_input': {'product_type': 'mortgage', 'date': '2025-01-15'}, 'tool_output': '변동금리 3.8~5.2%, 고정금리 4.5~5.5%'},
    ],
    'card': [
        {'tool_name': 'search_knowledge_base', 'tool_input': {'query': '카드 수수료 정책', 'top_k': 3}, 'tool_output': '검색 결과 2건 반환'},
        {'tool_name': 'get_card_benefits', 'tool_input': {'card_id': 'CC-PREMIUM-01'}, 'tool_output': '연회비, 포인트, 할인 정보 반환'},
        {'tool_name': 'lookup_fee_schedule', 'tool_input': {'fee_type': 'overseas_transaction'}, 'tool_output': '해외결제 수수료 1.0~1.5%'},
    ],
    'account': [
        {'tool_name': 'search_knowledge_base', 'tool_input': {'query': '계좌 관리 안내', 'top_k': 3}, 'tool_output': '검색 결과 3건 반환'},
        {'tool_name': 'get_account_policy', 'tool_input': {'policy_type': 'withdrawal_limit'}, 'tool_output': '1일 출금한도 300만원'},
        {'tool_name': 'check_service_availability', 'tool_input': {'service': 'mobile_banking', 'region': 'overseas'}, 'tool_output': '해외 이용 가능, 사전 등록 필요'},
    ],
}

DUMMY_NAMES   = ['김민수', '이지은', '박서준', '최유진', '정하늘', '강도윤', '윤서아', '한지호', '오채원', '임태현']
DUMMY_PHONES  = ['010-1234-5678', '010-9876-5432', '010-5555-1234', '010-3333-4444']
DUMMY_IDS     = ['110-234-567890', '352-0123-4567-13', '3333-01-1234567']  # 계좌번호
PII_ID_LABEL  = '계좌번호'

HALLUCINATION_TEMPLATES = [
    '{}에 대해 말씀드리면, 저희 은행은 업계 최초로 수수료 0원 정책을 시행하고 있습니다. 또한 금리 10%를 보장합니다.',
    '네, {} 관련하여 정부에서 최근 발표한 특별 지원금으로 100만원을 지급받으실 수 있습니다.',
    '{}에 대한 정보입니다. 현재 한시적으로 모든 수수료가 면제되며, 추가로 50만 포인트가 적립됩니다.',
]

WRONG_CONTEXTS = [
    '[(CAFETERIA_MENU_WEEKLY)]\n- 월: 된장찌개, 제육볶음\n- 화: 김치찌개, 고등어조림',
    '[(OFFICE_SUPPLY_INVENTORY)]\n- A4 용지: 50박스\n- 토너: 20개\n- 볼펜: 100개',
    '[(PARKING_LOT_RULES)]\n- 주차 가능 시간: 08:00~22:00\n- 요금: 30분당 1,000원',
]

PLANNING_TEMPLATES = [
    [
        {'step': 1, 'action': 'analyze_intent',   'thought': '사용자 질문의 의도를 파악합니다.'},
        {'step': 2, 'action': 'search_knowledge', 'thought': '관련 지식 베이스를 검색합니다.'},
        {'step': 3, 'action': 'generate_response','thought': '검색 결과를 바탕으로 답변을 생성합니다.'},
    ],
    [
        {'step': 1, 'action': 'classify_domain',     'thought': '질문이 어떤 도메인에 해당하는지 분류합니다.'},
        {'step': 2, 'action': 'retrieve_context',    'thought': '해당 도메인의 정책/문서를 검색합니다.'},
        {'step': 3, 'action': 'validate_information','thought': '검색된 정보의 최신성과 정확성을 확인합니다.'},
        {'step': 4, 'action': 'compose_answer',      'thought': '검증된 정보로 답변을 작성합니다.'},
    ],
    [
        {'step': 1, 'action': 'parse_query',     'thought': '질문에서 핵심 키워드와 조건을 추출합니다.'},
        {'step': 2, 'action': 'multi_tool_call', 'thought': '여러 도구를 호출하여 종합 정보를 수집합니다.'},
        {'step': 3, 'action': 'synthesize',      'thought': '수집된 정보를 종합하여 최종 답변을 구성합니다.'},
    ],
]

SCENARIO_DOMAINS = {d['domain'] for d in QA_POOL}

print(f'✓ 도메인 수: {len(QA_POOL)}개 → {sorted(SCENARIO_DOMAINS)}')
print(f'✓ 총 질문 풀: {sum(len(d["questions"]) for d in QA_POOL)}개')
print(f'✓ 도구 호출 템플릿: {sum(len(v) for v in TOOL_CALL_TEMPLATES.values())}개')
print(f'✓ 플래닝 템플릿: {len(PLANNING_TEMPLATES)}개')


✓ 도메인 수: 4개 → ['account', 'card', 'loan', 'savings']
✓ 총 질문 풀: 36개
✓ 도구 호출 템플릿: 12개
✓ 플래닝 템플릿: 3개


In [4]:
def inject_pii(text: str) -> str:
    """텍스트에 랜덤하게 PII를 삽입 (시나리오에 맞는 더미 사용)"""
    pii_type = random.choice(['name', 'phone', 'id', 'none', 'none', 'none'])
    if pii_type == 'name':
        return f'{random.choice(DUMMY_NAMES)}입니다. {text}'
    if pii_type == 'phone':
        return f'{text} 연락처는 {random.choice(DUMMY_PHONES)}입니다.'
    if pii_type == 'id':
        return f'{PII_ID_LABEL} {random.choice(DUMMY_IDS)} 관련 문의입니다. {text}'
    return text


def generate_raw_logs(n: int = 500, qa_pool: list = None) -> list:
    """운영 로그(Agent trace) 시뮬레이션 생성

    Parameters
    ----------
    n        : 생성할 로그 건수
    qa_pool  : 사용할 도메인 풀 (None이면 현재 시나리오의 QA_POOL 사용)
    """
    pool = qa_pool if qa_pool is not None else QA_POOL

    logs = []
    base_time = datetime(2025, 1, 1, 9, 0, 0)
    session_counter = 0

    while len(logs) < n:
        session_counter += 1
        session_id = f'sess_{session_counter:05d}'
        user_id = f'user_{random.randint(1, 200):04d}'
        domain_data = random.choice(pool)
        domain = domain_data['domain']

        # 멀티턴 대화 (1~3턴)
        num_turns = random.choices([1, 2, 3], weights=[0.6, 0.3, 0.1])[0]

        for turn in range(num_turns):
            if len(logs) >= n:
                break

            timestamp = base_time + timedelta(
                days=random.randint(0, 90),
                hours=random.randint(0, 14),
                minutes=random.randint(0, 59),
                seconds=random.randint(0, 59),
            )

            q_idx = random.randint(0, len(domain_data['questions']) - 1)
            question = domain_data['questions'][q_idx]

            # PII 삽입 (약 30%)
            if random.random() < 0.3:
                question = inject_pii(question)

            # 응답 유형 결정
            response_type = random.choices(
                ['good', 'hallucination', 'rag_fail', 'error', 'incomplete'],
                weights=[0.55, 0.12, 0.10, 0.08, 0.15],
            )[0]

            model = random.choice(['gpt-4o', 'gpt-4o-mini', 'gpt-4.1'])
            record = {
                'timestamp': timestamp.isoformat(),
                'session_id': session_id,
                'turn_index': turn,
                'user_id': user_id,
                'user_message': question,
                'model_name': model,
                'domain': domain,
            }

            # Planning steps (약 70%)
            if random.random() < 0.7:
                record['planning_steps'] = random.choice(PLANNING_TEMPLATES)

            # Tool calls (약 80%)
            if random.random() < 0.8:
                domain_tools = TOOL_CALL_TEMPLATES.get(domain, [])
                if domain_tools:
                    num_calls = random.randint(1, min(3, len(domain_tools)))
                    selected_tools = random.sample(domain_tools, num_calls)
                    tool_calls = []
                    for tool in selected_tools:
                        tc = tool.copy()
                        tc['call_id'] = f'call_{random.randint(10000, 99999)}'
                        tc['duration_ms'] = random.randint(50, 800)
                        if random.random() < 0.05:
                            tc['tool_output'] = 'ERROR: Timeout after 30s'
                            tc['status'] = 'failed'
                        else:
                            tc['status'] = 'success'
                        tool_calls.append(tc)
                    record['tool_calls'] = tool_calls

            # 토큰 사용량
            input_tokens = random.randint(150, 800)
            output_tokens = random.randint(50, 400)
            record['token_usage'] = {
                'input_tokens': input_tokens,
                'output_tokens': output_tokens,
                'total_tokens': input_tokens + output_tokens,
            }

            c_idx = min(q_idx, len(domain_data['contexts']) - 1)
            r_idx = min(q_idx, len(domain_data['good_responses']) - 1)

            if response_type == 'good':
                record['assistant_message'] = domain_data['good_responses'][r_idx]
                record['retrieved_context'] = domain_data['contexts'][c_idx]
                record['latency_ms'] = random.randint(200, 1500)
            elif response_type == 'hallucination':
                template = random.choice(HALLUCINATION_TEMPLATES)
                record['assistant_message'] = template.format(question[:10])
                record['retrieved_context'] = domain_data['contexts'][c_idx]
                record['latency_ms'] = random.randint(300, 2000)
            elif response_type == 'rag_fail':
                wrong = random.choice(WRONG_CONTEXTS)
                record['assistant_message'] = f'제공된 정보에서 관련 내용을 찾기 어렵습니다. {wrong[:30]}'
                record['retrieved_context'] = wrong
                record['latency_ms'] = random.randint(500, 3000)
            elif response_type == 'error':
                record['assistant_message'] = random.choice(['', None, 'ERROR: Internal Server Error'])
                record['retrieved_context'] = None
                record['latency_ms'] = random.choice([None, 0, 30000])
                record['error_flag'] = True
            elif response_type == 'incomplete':
                full_resp = domain_data['good_responses'][r_idx]
                record['assistant_message'] = full_resp[:len(full_resp) // 3] + '...'
                record['retrieved_context'] = domain_data['contexts'][c_idx]
                record['latency_ms'] = random.randint(5000, 15000)

            # 피드백 (~20%)
            if random.random() < 0.2:
                if response_type == 'good':
                    record['feedback_score'] = random.choices([4, 5], weights=[0.3, 0.7])[0]
                elif response_type in ('hallucination', 'rag_fail'):
                    record['feedback_score'] = random.choices([1, 2], weights=[0.6, 0.4])[0]
                else:
                    record['feedback_score'] = random.randint(1, 3)

            logs.append(record)

    # 의도적 중복 (재시도 시뮬레이션)
    duplicates = random.sample(logs[:min(400, len(logs))], min(20, len(logs)))
    for dup in duplicates:
        dup_copy = dup.copy()
        session_counter += 1
        dup_copy['session_id'] = f'sess_{session_counter:05d}'
        new_ts = datetime.fromisoformat(dup['timestamp']) + timedelta(minutes=random.randint(1, 60))
        dup_copy['timestamp'] = new_ts.isoformat()
        logs.append(dup_copy)

    return logs[:n + 20]


# ============================================================
# 🅰️ BYO trace  /  🅱️ 시뮬레이션 분기
# ============================================================
USE_EXISTING_LOG = os.getenv('USE_EXISTING_LOG', 'false').strip().lower() == 'true'
raw_log_path = DATA_DIR / 'raw_chat_logs.jsonl'

if USE_EXISTING_LOG and raw_log_path.exists():
    raw_logs = []
    with open(raw_log_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                raw_logs.append(json.loads(line))
    print(f'✓ BYO trace 로드: {len(raw_logs)}건  (시뮬레이션 스킵)')
    print(f'  source: {raw_log_path}')
else:
    raw_logs = generate_raw_logs(500)
    with open(raw_log_path, 'w', encoding='utf-8') as f:
        for log in raw_logs:
            f.write(json.dumps(log, ensure_ascii=False, default=str) + '\n')
    print(f'✓ 시뮬레이션 raw 로그 생성: {len(raw_logs)}건  (banking)')
    print(f'  saved : {raw_log_path}')

print('\n--- 샘플 로그 (1건) ---')
print(json.dumps(raw_logs[0], ensure_ascii=False, indent=2, default=str))


✓ 시뮬레이션 raw 로그 생성: 520건  (banking)
  saved : data/raw_chat_logs.jsonl

--- 샘플 로그 (1건) ---
{
  "timestamp": "2025-02-05T12:14:08",
  "session_id": "sess_00001",
  "turn_index": 0,
  "user_id": "user_0164",
  "user_message": "정기예금 만기 후 어떻게 되나요?",
  "model_name": "gpt-4o",
  "domain": "savings",
  "planning_steps": [
    {
      "step": 1,
      "action": "analyze_intent",
      "thought": "사용자 질문의 의도를 파악합니다."
    },
    {
      "step": 2,
      "action": "search_knowledge",
      "thought": "관련 지식 베이스를 검색합니다."
    },
    {
      "step": 3,
      "action": "generate_response",
      "thought": "검색 결과를 바탕으로 답변을 생성합니다."
    }
  ],
  "tool_calls": [
    {
      "tool_name": "search_knowledge_base",
      "tool_input": {
        "query": "적금 금리 정보",
        "top_k": 3
      },
      "tool_output": "ERROR: Timeout after 30s",
      "call_id": "call_76237",
      "duration_ms": 666,
      "status": "failed"
    }
  ],
  "token_usage": {
    "input_tokens": 353,
    "output_tokens": 382,
    "to

## 4단계: PII 제거 / 비식별화

운영 로그에는 고객 개인정보가 포함되어 있을 수 있습니다.  
평가용 데이터셋으로 사용하기 전에 **반드시 개인정보를 제거**해야 합니다.

| PII 유형 | 패턴 | 대체값 |
|---------|------|--------|
| 전화번호 | `010-XXXX-XXXX` | `[PHONE]` |
| 계좌번호 | `XXX-XXXX-XXXXXX` | `[ACCOUNT]` |
| 이름 (한글) | 2~4자 한글 + '입니다' | `[NAME]` |

> ⚠️ 실제 운영 환경에서는 Azure AI Content Safety, Presidio 등 전문 도구를 사용하는 것을 권장합니다.

In [5]:
def remove_pii(text: str) -> str:
    """텍스트에서 PII를 탐지하고 마스킹"""
    if not isinstance(text, str):
        return text
    
    # 전화번호 마스킹
    text = re.sub(r'01[016789]-\d{3,4}-\d{4}', '[PHONE]', text)
    
    # 계좌번호 마스킹 (다양한 형식)
    text = re.sub(r'\d{3}-\d{3,4}-\d{4,6}(-\d{2})?', '[ACCOUNT]', text)
    
    # 한글 이름 + '입니다' 패턴 마스킹
    text = re.sub(r'[가-힣]{2,4}입니다', '[NAME]입니다', text)
    
    return text


# PII 제거 적용
df_raw = pd.DataFrame(raw_logs)

pii_count_before = 0
for col in ['user_message', 'assistant_message']:
    mask = df_raw[col].astype(str).str.contains(r'01[016789]-\d{3,4}-\d{4}|\d{3}-\d{3,4}-\d{4,6}|[가-힣]{2,4}입니다', na=False)
    pii_count_before += mask.sum()

df_raw['user_message'] = df_raw['user_message'].apply(remove_pii)
df_raw['assistant_message'] = df_raw['assistant_message'].apply(remove_pii)

print(f'✓ PII 탐지 및 마스킹 완료')
print(f'  - PII 포함 레코드: {pii_count_before}건')

# PII 마스킹 결과 샘플
pii_examples = df_raw[df_raw['user_message'].str.contains(r'\[NAME\]|\[PHONE\]|\[ACCOUNT\]', na=False)].head(3)
if len(pii_examples) > 0:
    print('\n--- PII 마스킹 예시 ---')
    for _, row in pii_examples.iterrows():
        print(f'  → {row["user_message"]}')

✓ PII 탐지 및 마스킹 완료
  - PII 포함 레코드: 158건

--- PII 마스킹 예시 ---
  → 신용카드 포인트 사용법은? 연락처는 [PHONE]입니다.
  → 계좌번호 [ACCOUNT] 관련 [NAME]입니다. 마이너스 통장 개설 조건은?
  → 카드 결제일 변경 가능해? 연락처는 [PHONE]입니다.


## 5단계: 탐색적 데이터 분석 (EDA)

데이터의 품질과 분포를 파악하여 정제 전략을 수립합니다.

In [6]:
print('=' * 60)
print('📊 데이터 기본 통계')
print('=' * 60)
print(f'총 레코드 수: {len(df_raw):,}건')
print(f'고유 세션 수: {df_raw["session_id"].nunique():,}개')
print(f'고유 사용자 수: {df_raw["user_id"].nunique():,}명')
print(f'기간: {df_raw["timestamp"].min()} ~ {df_raw["timestamp"].max()}')

print(f'\n--- 도메인 분포 ---')
print(df_raw['domain'].value_counts().to_string())

print(f'\n--- 모델 분포 ---')
print(df_raw['model_name'].value_counts().to_string())

print(f'\n--- 결측값 현황 ---')
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
missing_df = pd.DataFrame({'결측 수': missing, '비율(%)': missing_pct})
print(missing_df[missing_df['결측 수'] > 0].to_string())

print(f'\n--- 오류 레코드 ---')
error_count = df_raw.get('error_flag', pd.Series(dtype=bool)).sum()
print(f'오류 플래그 레코드: {int(error_count) if not pd.isna(error_count) else 0}건')

print(f'\n--- 응답 길이 통계 ---')
df_raw['response_length'] = df_raw['assistant_message'].astype(str).str.len()
print(df_raw['response_length'].describe().to_string())

print(f'\n--- 사용자 피드백 통계 ---')
if 'feedback_score' in df_raw.columns:
    fb = df_raw['feedback_score'].dropna()
    print(f'피드백 있는 레코드: {len(fb)}건 ({len(fb)/len(df_raw)*100:.1f}%)')
    print(f'평균 점수: {fb.mean():.2f}')
    print(fb.value_counts().sort_index().to_string())

📊 데이터 기본 통계
총 레코드 수: 520건
고유 세션 수: 349개
고유 사용자 수: 164명
기간: 2025-01-01T11:52:31 ~ 2025-04-01T19:30:10

--- 도메인 분포 ---
domain
card       148
account    133
loan       128
savings    111

--- 모델 분포 ---
model_name
gpt-4o-mini    184
gpt-4.1        173
gpt-4o         163

--- 결측값 현황 ---
                   결측 수  비율(%)
planning_steps      162   31.2
tool_calls          104   20.0
assistant_message    10    1.9
retrieved_context    33    6.3
latency_ms           11    2.1
feedback_score      407   78.3
error_flag          487   93.7

--- 오류 레코드 ---
오류 플래그 레코드: 33건

--- 응답 길이 통계 ---
count    520.000000
mean      53.534615
std       21.217989
min        0.000000
25%       42.000000
50%       60.000000
75%       68.000000
max       92.000000

--- 사용자 피드백 통계 ---
피드백 있는 레코드: 113건 (21.7%)
평균 점수: 3.27
feedback_score
1.0    21
2.0    24
3.0    11
4.0    17
5.0    40


## 6단계: 데이터 정제 (Cleaning)

EDA 결과를 바탕으로 평가에 적합하지 않은 데이터를 제거합니다.

### 정제 기준

| 기준 | 제거 조건 | 사유 |
|------|----------|------|
| 오류 응답 | `error_flag == True` 또는 응답이 비어있음 | 평가 대상이 아님 |
| 응답 길이 | 10자 미만 | 의미 있는 평가 불가 |
| 컨텍스트 누락 | `retrieved_context`가 None | Retrieval/Groundedness 평가 불가 |
| 근접 중복 | 동일 사용자 + 동일 질문 (정규화 후) | 평가 편향 방지 |

In [7]:
df_clean = df_raw.copy()
initial_count = len(df_clean)
removal_log = []

# 1. 오류 응답 제거
error_mask = (
    (df_clean.get('error_flag', False) == True) |
    (df_clean['assistant_message'].isna()) |
    (df_clean['assistant_message'].astype(str).isin(['', 'None', 'ERROR: Internal Server Error']))
)
removed = error_mask.sum()
df_clean = df_clean[~error_mask]
removal_log.append(('오류 응답', removed))

# 2. 응답 길이 필터
short_mask = df_clean['assistant_message'].astype(str).str.len() < 10
removed = short_mask.sum()
df_clean = df_clean[~short_mask]
removal_log.append(('짧은 응답 (10자 미만)', removed))

# 3. 컨텍스트 누락 제거
ctx_mask = df_clean['retrieved_context'].isna() | (df_clean['retrieved_context'].astype(str) == 'None')
removed = ctx_mask.sum()
df_clean = df_clean[~ctx_mask]
removal_log.append(('컨텍스트 누락', removed))

# 4. 근접 중복 제거 (동일 user + 질문 정규화 비교)
df_clean['_norm_question'] = (
    df_clean['user_message']
    .str.replace(r'\[NAME\]입니다\.\s*', '', regex=True)
    .str.replace(r'연락처는 \[PHONE\]입니다\.', '', regex=True)
    .str.replace(r'계좌번호 \[ACCOUNT\] 관련 문의입니다\.\s*', '', regex=True)
    .str.strip()
)
before_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['user_id', '_norm_question'], keep='first')
removed = before_dedup - len(df_clean)
removal_log.append(('근접 중복', removed))
df_clean = df_clean.drop(columns=['_norm_question', 'response_length'], errors='ignore')

# 정제 결과 요약
print('=' * 60)
print('🧹 데이터 정제 결과')
print('=' * 60)
for reason, count in removal_log:
    print(f'  - {reason}: {count}건 제거')
print(f'\n  원본: {initial_count}건 → 정제 후: {len(df_clean)}건 ({len(df_clean)/initial_count*100:.1f}%)')

🧹 데이터 정제 결과
  - 오류 응답: 33건 제거
  - 짧은 응답 (10자 미만): 0건 제거
  - 컨텍스트 누락: 0건 제거
  - 근접 중복: 40건 제거

  원본: 520건 → 정제 후: 447건 (86.0%)


## 7단계: 대화 → 턴 단위 변환

평가 SDK의 Evaluator는 **단일 턴(query-response 쌍)** 단위로 동작합니다.  
멀티턴 대화 로그에서 각 턴을 독립적인 평가 단위로 추출합니다.

### 변환 규칙

- 각 턴은 `(user_message, assistant_message, retrieved_context)` 1행으로 변환
- 후속 질문(follow-up)은 문맥 독립적인 형태로 보존 (이 데모에서는 각 턴이 이미 독립 질문)
- 원본 `session_id`와 `turn_index`를 추적 ID로 유지

### 실무적으로 여기서 추가로 만드는 값: `weak_outcome`

`weak_outcome`은 **정답 라벨이 아니라**, 나중 단계에서 "어떤 로그를 먼저 보고, 어떤 실패를 더 뽑아볼지" 판단하기 위한 **임시 분류 태그**입니다.

- `good`: 겉보기엔 큰 이상이 없어 보이는 응답
- `hallucination`: 이상한 사실 주장 가능성이 있는 응답
- `rag_fail`: 검색 실패처럼 보이는 응답
- `incomplete`: 답변이 중간에 잘린 것처럼 보이는 응답

즉, 여기서는 **정답/오답을 확정하는 것이 아니라, 검토 우선순위를 잡기 위한 표시**만 만든다고 이해하면 됩니다.

In [8]:
# 턴 단위 데이터프레임 구성
df_turns = df_clean.copy()

# 추적 ID 생성
df_turns['source_id'] = df_turns['session_id'] + '_t' + df_turns['turn_index'].astype(str)

# ============================================================
# weak_outcome 부여 (7.5 샘플링/8.5 검수 우선순위에 사용하는 약식 라벨)
#   - good          : 구조적으로 큰 이상이 보이지 않는 응답
#   - hallucination : 컨텍스트에 없는 사실 주장 가능성 (휴리스틱 키워드 기반)
#   - rag_fail      : 검색 실패형 응답 가능성
#   - incomplete    : 응답 잘림 가능성 ('...' 종료)
#
# 중요: 이 값은 정답 라벨이 아니라 샘플링과 검수 우선순위를 위한 weak signal입니다.
# 공식 평가는 8.5단계의 LLM-assisted label + human review를 거친 Gold Set에서 수행합니다.
# ============================================================
HALLUC_KEYS = ['수수료 0원', '특별 지원금', '한시적', '무재해 365일', '항상 99%', '영구적', '0건']

def _infer_weak_outcome(row):
    msg = str(row.get('assistant_message', ''))
    if msg.endswith('...'):
        return 'incomplete'
    if msg.startswith('제공된 정보에서 관련 내용을 찾기 어렵습니다'):
        return 'rag_fail'
    if any(k in msg for k in HALLUC_KEYS):
        return 'hallucination'
    return 'good'

df_turns['weak_outcome'] = df_turns.apply(_infer_weak_outcome, axis=1)

print(f'✓ 턴 단위 데이터: {len(df_turns)}건')
print(f'  - 세션 수: {df_turns["session_id"].nunique()}개')
print(f'  - 턴 분포:')
print(df_turns['turn_index'].value_counts().sort_index().to_string())
print(f'\n  - 도메인 분포:')
print(df_turns['domain'].value_counts().to_string())
print(f'\n  - weak_outcome 분포 (휴리스틱 기반 샘플링용 추정치):')
print(df_turns['weak_outcome'].value_counts().to_string())


✓ 턴 단위 데이터: 447건
  - 세션 수: 317개
  - 턴 분포:
turn_index
0    305
1    112
2     30

  - 도메인 분포:
domain
card       123
loan       119
account    109
savings     96

  - weak_outcome 분포 (휴리스틱 기반 샘플링용 추정치):
weak_outcome
good             260
incomplete        82
hallucination     58
rag_fail          47


## 7.5단계 ⭐ 평가 후보 뽑기 (쉽게 말해, 무엇을 먼저 볼지 정하는 단계)

> **이 단계에서 답하려는 질문**: "운영 로그가 많을 때, 어떤 로그를 먼저 평가 후보로 가져오면 되나?"

### 먼저 아주 간단히

이 단계는 **최종 점수를 계산하는 단계가 아닙니다.**  
이 단계에서 하는 일은 딱 하나입니다.

**전체 로그 중에서, 나중에 사람이 보거나 LLM judge를 돌려볼 만한 후보를 잘 뽑는 것**

### 실무자 관점에서 해야 하는 액션

1. 전체 운영 로그를 그대로 다 보지 않는다.
2. 도메인이 한쪽으로 치우치지 않게 골고루 뽑는다.
3. 정상처럼 보이는 로그만 뽑지 말고, 실패처럼 보이는 로그도 일부러 더 뽑는다.
4. 비슷한 질문만 반복해서 뽑지 않도록 중복성 높은 질문은 줄인다.
5. 이렇게 만든 후보셋을 다음 단계(8.5)에서 레이블링/검수한다.

### 이 노트북이 실제로 하는 일

| 쉬운 설명 | 노트북에서 하는 일 | 왜 필요한가 |
|---|---|---|
| 몇 건 정도 볼지 정함 | Cochran 공식으로 후보 수 계산 | 너무 적게 뽑지 않기 위해 |
| 도메인을 골고루 섞음 | `domain × weak_outcome` 기준 quota 추출 | 특정 도메인에 쏠리지 않기 위해 |
| 실패 케이스를 더 확보함 | `hallucination`, `rag_fail`, `incomplete` 비중을 일부러 높임 | 실제 문제 유형을 분석하기 위해 |
| 비슷한 질문을 줄임 | TF-IDF + KMeans로 대표 질문만 선택 | 같은 질문만 반복 평가하지 않기 위해 |

### 여기서 `weak_outcome`을 왜 쓰나

운영 로그를 무작위로만 뽑으면, 겉보기 정상 응답이 너무 많이 뽑히고 실패 케이스는 거의 안 잡힐 수 있습니다.  
그래서 `weak_outcome`이라는 **임시 실패 추정 태그**를 이용해 실패 가능성이 있는 로그를 더 섞어 뽑습니다.

중요한 점은 다음입니다.

- `weak_outcome`은 **정답이 아님**
- 단지 **후보를 더 잘 뽑기 위한 힌트**임
- 공식 점수는 나중에 사람 검수된 데이터로 계산해야 함

### 통계를 잘 몰라도 이렇게 이해하면 충분합니다

- Cochran: "후보를 너무 적게 뽑지 않기 위한 기준값"
- quota sampling: "도메인과 실패 유형이 한쪽으로 몰리지 않게 뽑기"
- diversity sampling: "비슷한 질문은 줄이고 대표 질문만 남기기"

### 이 단계의 결과

이 단계가 끝나면 **최종 평가셋이 아니라 `평가 후보셋(df_sampled)`** 이 만들어집니다.  
이 후보셋을 다음 단계에서 LLM-assisted labeling과 사람 검수 대상으로 사용합니다.

### ⚠️ 꼭 기억할 점

- 여기서 나온 비율은 공식 품질 점수가 아닙니다.
- 여기서 하는 일은 **좋은 후보를 고르는 것**입니다.
- 실제 의사결정용 평가는 8.5단계 이후의 사람 검수 데이터로 해야 합니다.

In [9]:
from scipy import stats as scipy_stats
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# ------------------------------------------------------------
# 1) Cochran 표본 크기 산정
# ------------------------------------------------------------
def cochran_sample_size(N: int, confidence: float = 0.95,
                        margin: float = 0.05, p: float = 0.5) -> int:
    """유한모집단 보정 포함 Cochran 후보 표본 크기."""
    z = scipy_stats.norm.ppf(1 - (1 - confidence) / 2)
    n0 = (z ** 2 * p * (1 - p)) / (margin ** 2)
    n = n0 / (1 + (n0 - 1) / max(N, 1))
    return int(np.ceil(n))


N_total = len(df_turns)
n_target = cochran_sample_size(N_total)
print(f'운영 turn 모집단 N = {N_total}')
print(f'95% CL ±5% 기준 후보 표본 목표 n_target = {n_target}  (Cochran, p=0.5 보수)')

# 운영 N이 작은 데모 환경에선 표본이 모집단을 초과할 수 있음 → 캡
n_target = min(n_target, N_total)

# ------------------------------------------------------------
# 2) 구조적 신호 + weak_outcome 기반 quota sampling
#    - domain: 관측 가능한 구조적 축
#    - weak_outcome: 휴리스틱/LLM judge로 대체 가능한 약식 위험 라벨
# ------------------------------------------------------------
WEAK_OUTCOME_WEIGHTS = {
    'good':          0.45,
    'hallucination': 0.20,
    'rag_fail':      0.20,
    'incomplete':    0.15,
}

def quota_sample(
    df: pd.DataFrame,
    n: int,
    weak_outcome_weights: dict,
    stratify_col: str = 'domain',
    weak_label_col: str = 'weak_outcome',
    random_state: int = RANDOM_SEED,
) -> pd.DataFrame:
    """domain × weak_outcome quota sampling.

    weak_outcome은 진실 라벨이 아니라 샘플링과 검수 우선순위를 위한 weak signal입니다.
    공식 metric은 이후 human-reviewed Gold/Test/Holdout에서 계산해야 합니다.
    """
    pieces = []
    strata = sorted(df[stratify_col].dropna().unique())
    total_w = sum(weak_outcome_weights.values())

    for weak_label, weight in weak_outcome_weights.items():
        bucket = df[df[weak_label_col] == weak_label]
        if len(bucket) == 0:
            continue
        target = int(round(n * weight / total_w))
        per_stratum = max(1, target // max(len(strata), 1))
        for value in strata:
            sub = bucket[bucket[stratify_col] == value]
            if len(sub) == 0:
                continue
            take = min(per_stratum, len(sub))
            pieces.append(sub.sample(n=take, random_state=random_state))

    if not pieces:
        return df.head(0).copy()
    return pd.concat(pieces).drop_duplicates(subset=['source_id']).reset_index(drop=True)


df_quota = quota_sample(df_turns, n_target, WEAK_OUTCOME_WEIGHTS)
print(f'\n[Screening Sample] domain × weak_outcome 후보 추출 후 = {len(df_quota)}건')
print(df_quota.groupby(['domain', 'weak_outcome']).size().unstack(fill_value=0))

# ------------------------------------------------------------
# 3) 다양성 샘플링 (TF-IDF + KMeans → 클러스터당 대표 K건)
# ------------------------------------------------------------
def diversity_sample(
    df: pd.DataFrame,
    max_per_cluster: int = 2,
    text_col: str = 'user_message',
    weak_label_col: str = 'weak_outcome',
    random_state: int = RANDOM_SEED,
) -> pd.DataFrame:
    pieces = []
    for weak_label, sub in df.groupby(weak_label_col):
        if len(sub) < 4:
            pieces.append(sub)
            continue
        try:
            vec = TfidfVectorizer(max_features=200).fit_transform(
                sub[text_col].fillna('').astype(str)
            )
            n_clusters = max(2, min(len(sub) // 2, 10))
            labels = KMeans(
                n_clusters=n_clusters,
                random_state=random_state,
                n_init=5,
            ).fit_predict(vec)
        except ValueError:
            pieces.append(sub)
            continue
        sub = sub.copy()
        sub['_cluster'] = labels
        rep = (
            sub.groupby('_cluster', group_keys=False)
            .apply(lambda x: x.sample(n=min(len(x), max_per_cluster), random_state=random_state))
        )
        pieces.append(rep.drop(columns='_cluster'))
    return pd.concat(pieces).reset_index(drop=True)


df_sampled = diversity_sample(df_quota)
print(f'\n[Diversity] 유사 질문 클러스터 대표 추출 후 후보 = {len(df_sampled)}건')
if len(df_sampled) < n_target:
    print(f'⚠️ 최종 후보({len(df_sampled)})가 목표 후보({n_target})보다 작습니다. '
          'max_per_cluster를 늘리거나 quota 비율을 조정하세요.')

# ------------------------------------------------------------
# 4) 검증 — 분포 / 누락 도메인 / weak_outcome oversample 효과
# ------------------------------------------------------------
print('\n=== 최종 평가 후보셋 분포 ===')
print('\n[domain × weak_outcome]')
print(df_sampled.groupby(['domain', 'weak_outcome']).size().unstack(fill_value=0))
print('\n[운영(turns) vs 후보(sampled) weak_outcome 비율 비교]')
ops = df_turns['weak_outcome'].value_counts(normalize=True).round(3)
smp = df_sampled['weak_outcome'].value_counts(normalize=True).round(3)
print(pd.DataFrame({'ops_weak_estimate': ops, 'sampled_candidate': smp}).fillna(0))

# 누락 도메인 경고
missing_domains = set(df_turns['domain'].unique()) - set(df_sampled['domain'].unique())
if missing_domains:
    print(f'\n⚠️  후보 표본에서 누락된 도메인: {missing_domains}')
else:
    print('\n✓ 모든 도메인이 후보 표본에 포함됨')

print('\n주의: 이 단계의 분포는 공식 품질 점수가 아니라, 8.5단계 LLM-assisted labeling과 human review로 보정할 후보 표본 분포입니다.')


운영 turn 모집단 N = 447
95% CL ±5% 기준 후보 표본 목표 n_target = 207  (Cochran, p=0.5 보수)

[Screening Sample] domain × weak_outcome 후보 추출 후 = 195건
weak_outcome  good  hallucination  incomplete  rag_fail
domain                                                 
account         23             10           7        10
card            23             10           7        10
loan            23             10           7        10
savings         23              9           7         6

[Diversity] 유사 질문 클러스터 대표 추출 후 후보 = 80건
⚠️ 최종 후보(80)가 목표 후보(207)보다 작습니다. max_per_cluster를 늘리거나 quota 비율을 조정하세요.

=== 최종 평가 후보셋 분포 ===

[domain × weak_outcome]
weak_outcome  good  hallucination  incomplete  rag_fail
domain                                                 
account          7              5           4         5
card             7              7           6         6
loan             3              2           7         7
savings          3              6           3         2

[운영(turns) vs 후보(sampled) weak_

/var/folders/0r/wx9n6dq14758w8v_tc4rdpbc0000gn/T/ipykernel_41837/2325003152.py:108: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), max_per_cluster), random_state=random_state))
/var/folders/0r/wx9n6dq14758w8v_tc4rdpbc0000gn/T/ipykernel_41837/2325003152.py:108: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), max_per_cluster), random_state=ran

## 8단계: 평가 스키마 변환

정제된 데이터를 Azure AI Evaluation SDK가 요구하는 형식으로 변환합니다.

### 평가 데이터 유형

| 유형 | 설명 | Ground Truth | 용도 |
|------|------|-------------|------|
| **Silver Set** | 자동 변환, 레이블 없음 | ❌ | Retrieval, Groundedness 평가 |
| **Gold Set** | 수동 검수 + 정답 포함 | ✅ | QA, Custom Evaluator 평가 |

> 💡 **Silver Set**은 바로 사용 가능하지만, **Gold Set**은 도메인 전문가의 검수(Human Review)가 필요합니다.  
> 이 노트북에서는 자동 초안 생성 후 검수 대상을 표시하는 방식을 사용합니다.

> 📦 **왜 레이블링이 필요한가 (의사결정 관점)**  
> 레이블이 없으면 Retrieval/Groundedness 같은 "참고 지표" 중심으로만 볼 수 있어 **릴리즈/롤백 판단 근거가 약합니다**.  
> 레이블이 있으면 QA 정확도, 커스텀 pass/fail, tool trajectory 정합성을 계산해 **배포 게이트와 회귀 판정**에 직접 사용할 수 있습니다.

In [ ]:
def map_to_eval_schema(df: pd.DataFrame) -> pd.DataFrame:
    """
    운영 로그(샘플링 후) → 평가 스키마

    매핑 규칙:
      - user_message       → query
      - assistant_message  → response
      - retrieved_context  → context
      - weak_outcome       → weak_outcome  (샘플링/검수 우선순위용 약식 라벨)
    """
    eval_df = pd.DataFrame({
        'query': df['user_message'],
        'response': df['assistant_message'],
        'context': df['retrieved_context'],
        'domain': df['domain'],
        'source_id': df['source_id'],
        'session_id': df['session_id'],
        'turn_index': df['turn_index'],
        'model_name': df['model_name'],
        'weak_outcome': df['weak_outcome'],
    })

    if 'feedback_score' in df.columns:
        eval_df['feedback_score'] = df['feedback_score']

    return eval_df


# ⚠️ 입력은 df_turns 전체가 아니라 7.5단계에서 통계적으로 추려낸 후보 표본 df_sampled
# weak_outcome은 공식 정답 라벨이 아니라, 8.5단계 레이블링/검수 우선순위에 사용하는 약식 신호입니다.
df_eval = map_to_eval_schema(df_sampled)
print(f'✓ 평가 후보 스키마 변환 완료: {len(df_eval)}건  (샘플링 후보 기준)')
print(f'\n--- 스키마 ---')
print(df_eval.dtypes.to_string())
print(f'\n--- 샘플 (1건) ---')
print(json.dumps(df_eval.iloc[0].dropna().to_dict(), ensure_ascii=False, indent=2, default=str))


✓ 평가 후보 스키마 변환 완료: 80건  (샘플링 후보 기준)

--- 스키마 ---
query              object
response           object
context            object
domain             object
source_id          object
session_id         object
turn_index          int64
model_name         object
weak_outcome       object
feedback_score    float64

--- 샘플 (1건) ---
{
  "query": "통장 비밀번호 변경은?",
  "response": "압류된 계좌는 압류 금액 초과분에 대해 출금이 가능합니다. 생계비(월 185만원)는 법원에 압류범위변경 신청을 통해 보호받을 수 있습니다.",
  "context": "[(ACCOUNT_SEIZURE_FAQ)]\n- 압류 금액 초과분 출금 가능\n- 생계비(월 185만원) 보호 신청 가능\n- 법원 압류범위변경 신청 필요\n- 압류 통지: SMS 및 등기우편으로 통보\n- 압류 사유 확인: 법원/세무서 등 압류기관 문의\n- 해제 절차: 채무 변제 → 압류기관 해제 신청 → 은행 통보\n- 해제 처리 기간: 통상 3~5영업일\n- 임금/연금 압류: 1/2 보호 (민사집행법)\n- 보장성 보험 환급금: 일부 압류 제한\n- 미성년자 명의 계좌: 친권자 동의 시에만 압류 가능\n- 압류 중 신규 입금: 즉시 압류 대상 (생계비 제외)\n- 변호사/법무사 상담 권장",
  "domain": "account",
  "source_id": "sess_00086_t0",
  "session_id": "sess_00086",
  "turn_index": 0,
  "model_name": "gpt-4o",
  "weak_outcome": "good"
}


### Ground Truth 레이블링 (8.5단계)

> **현업이 가장 어려워하는 지점**: "정답이 없는 운영 로그에 어떻게 정답을 붙이나?"

### 이 단계의 역할

7.5단계가 **후보 표본을 만드는 단계**였다면, 8.5단계는 **그 후보에 신뢰 가능한 기준을 붙여 최종 평가셋으로 확정하는 단계**입니다.

즉, 실무 관점에서는 다음 두 단계를 분리해서 봐야 합니다.

1. **샘플링**: 전체 trace 중 어떤 로그를 평가 후보로 볼지 선택
2. **레이블링/보정**: 선택된 후보에 ground truth / confidence / review priority를 부여하고, 자동 라벨의 품질을 점검

### 이 단계가 통계적으로 중요한 이유

레이블이 없는 운영 로그에서는 7.5단계의 weak label만으로 공식 평가 숫자를 만들 수 없습니다.  
8.5단계가 필요한 이유는 다음과 같습니다.

- 자동 라벨은 빠르지만 오류가 있습니다.
- LLM judge는 일관성은 줄 수 있지만, 진실의 대체재는 아닙니다.
- 일부 샘플은 반드시 사람이 검수해야 judge-human agreement와 threshold를 확인할 수 있습니다.
- 결국 **공식 점수는 사람 검수된 Gold/Test/Holdout 세트에서 나와야** 합니다.

즉, 이 단계는 단순 레이블링이 아니라 **calibration 단계**이기도 합니다.

### 실무형 기본 프레임워크

가장 실무적인 기본 흐름은 아래와 같습니다.

```text
[1차 후보 표본]
    ↓
[LLM-assisted 초안 생성]
    ↓
[LLM judge / 규칙 기반 1차 신뢰도 부여]
    ↓
[사람 검수]
  - low confidence 우선
  - 중요 시나리오 우선
  - 배포 영향 큰 케이스 우선
    ↓
[최종 평가셋 확정]
  - Gold Set: 사람 검수 완료
  - Silver Set: 자동 생성 + 검수 대기
```

즉, **LLM이 최종 정답을 결정하는 것이 아니라, 사람 검수 비용을 줄이기 위한 초안 생성기 + 우선순위 분류기 역할**을 합니다.

### 예시: 1,500건 중 30%를 먼저 본다면

예를 들어 전체 1,500건 중 약 30%인 450건을 1차 후보로 잡았다고 하면, 실무적으로는 아래처럼 운영할 수 있습니다.

1. 450건에 대해 LLM judge로 약식 레이블을 붙인다.
2. confidence가 낮거나 위험도가 높은 케이스를 review queue로 보낸다.
3. 사람이 상위 위험군부터 검수한다.
4. 검수 완료분을 Gold Set으로 확정한다.
5. 검수 전 자동 라벨은 Silver Set으로 유지한다.

### Gold / Silver / Review Queue 운영 기준

| 구분 | 의미 | 언제 쓰나 | 신뢰 수준 | 운영 원칙 |
|---|---|---|---|---|
| **Gold Set** | 사람 검수까지 완료된 평가셋 | 공식 점수, 배포 게이트, 회귀 비교 | 높음 | Test/Holdout 중심으로 고정 관리 |
| **Silver Set** | 자동 생성 또는 약식 라벨 기반 평가셋 | 탐색 분석, 초기 evaluator 튜닝, triage | 중간~낮음 | 참고 지표로만 사용, 필요 시 Gold로 승격 |
| **Review Queue** | 사람 검수가 필요한 후보 목록 | low confidence, high risk, 중요 케이스 우선 검토 | 미확정 | 우선순위 기준을 명확히 두고 순차 검수 |

실무에서는 보통 아래처럼 운영합니다.

- **Gold Set**: 사람 검수 완료, 정답 확정, 배포 의사결정용
- **Silver Set**: 자동 라벨 기반, 빠른 탐색/실험용
- **Review Queue**: low confidence, hallucination 의심, 정책 민감, VIP/고객영향 큰 케이스 우선

즉, 모든 데이터를 처음부터 Gold로 만들기보다, **Silver와 Review Queue를 통해 검수 비용을 집중하고, 중요한 것만 Gold로 승격**하는 방식이 현실적입니다.

### Calibration 관점에서 꼭 봐야 하는 지표

사람 검수 표본이 생기면, 최소한 아래는 확인해야 합니다.

- **Judge-human agreement**: 자동 라벨과 사람 판단이 얼마나 일치하는가
- **False pass rate**: judge가 통과시켰지만 사람은 실패로 본 비율
- **False fail rate**: judge가 실패로 봤지만 사람은 통과로 본 비율
- **Threshold calibration**: 어떤 점수 cutoff에서 review queue를 보내는 것이 적절한가

이 지표가 없는 상태에서 Silver 점수를 공식 KPI처럼 해석하면 위험합니다.

### Rubric (정답 정의)

```
[GT 작성 규칙]
1. 정답은 사용자 질문에 직접 답하는 1~3 문장이어야 한다 (컨텍스트 발췌가 아니라 답변).
2. 컨텍스트에 명시된 사실에만 근거한다. 추론·외부지식 금지.
3. 수치/정책 조건은 컨텍스트의 표현을 우선 사용한다.
4. 컨텍스트로 답할 수 없으면 정확히 "정보 부족"으로 표기 (needs_review=True).
5. 부분 정답 인정: 핵심 사실 누락 없으면 high, 일부 누락은 medium, 사실 충돌은 low.
```

### 파이프라인

```
[A] LLM-assisted 초안 생성
       (Azure OpenAI 설정 시 — 미설정 시 휴리스틱 fallback)
        ↓
[B] LLM-judge agreement 점수 → confidence (high/medium/low)
        ↓
[C] 검수 우선순위 큐 (저신뢰 + weak_outcome 위험 신호 우선)
        ↓
[D] 도메인 전문가 사람 검수 (실무 단계 — 본 노트북 외부)
        ↓
[E] (Agent용) expected_tool_calls 정답 시퀀스 레이블
```

### Inter-Annotator Agreement (IAA)

2명 이상이 같은 샘플에 confidence 등급을 매기면 합의도를 측정해야 합니다.

- `sklearn.metrics.cohen_kappa_score(rater1, rater2)` (2명)
- `krippendorff` 패키지의 alpha (3명 이상)
- 일반 기준: κ ≥ 0.6 (substantial), κ ≥ 0.8 (almost perfect)
- κ가 낮으면 → Rubric이 모호하다는 신호. 가이드라인을 먼저 정교화.

### Agent 평가용 추가 레이블

RAG QA 정답(`ground_truth`)만으로는 **Tool-calling Agent**를 평가할 수 없습니다. 06/08 노트북이 사용하는 ToolCallAccuracy/IntentResolution 등은 **expected_tool_calls** (호출되어야 하는 tool 시퀀스)를 정답으로 요구합니다. 본 노트북은 `weak_outcome=='good'` 으로 추정되고 실제 성공 tool call이 있는 케이스의 호출을 expected_tool_calls 후보로 채택합니다. 이 역시 자동 후보이므로 실무에서는 사람 검수가 필요합니다.


In [ ]:
# ============================================================
# Ground Truth 레이블링 / 검수 큐 생성
# ============================================================
GT_RUBRIC = '''
[정답 레이블링 Rubric]
- 정답은 사용자 질문에 직접 답하는 1~3 문장이어야 한다 (컨텍스트 발췌가 아닌 답변).
- 컨텍스트에 명시된 사실에만 근거한다. 추론/외부지식 금지.
- 수치/정책 조건은 컨텍스트의 표현을 우선한다.
- 컨텍스트로 답할 수 없으면 정확히 "정보 부족" 으로 표기 (needs_review=True).
- 부분 정답 인정: 핵심 사실 누락 없으면 high, 일부 누락 medium, 사실 충돌 low.
'''
print(GT_RUBRIC)

USE_LLM_LABELER = bool(
    os.environ.get('AZURE_OPENAI_ENDPOINT') and os.environ.get('AZURE_OPENAI_API_KEY')
)
print(f'LLM 레이블러 사용 가능: {USE_LLM_LABELER}')
if not USE_LLM_LABELER:
    print('⚠️ LLM 레이블러가 꺼져 있어 fallback 모드로 동작합니다. 이 경우 confidence=low(검수 필요) 비율이 매우 높거나 100%가 되는 것이 정상입니다.')

_llm = None
_DEPLOY = os.environ.get('AZURE_OPENAI_DEPLOYMENT', 'gpt-4o-mini')

if USE_LLM_LABELER:
    try:
        from openai import AzureOpenAI
        _llm = AzureOpenAI(
            azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
            api_key=os.environ['AZURE_OPENAI_API_KEY'],
            api_version=os.environ.get('AZURE_OPENAI_API_VERSION', '2024-10-21'),
        )
    except Exception as e:
        print(f'⚠️ AzureOpenAI 클라이언트 초기화 실패 → 휴리스틱 fallback. ({type(e).__name__}: {e})')
        USE_LLM_LABELER = False


def _llm_chat(messages, max_tokens=200):
    """GPT-5/o-시리즈 reasoning 모델까지 호환되는 호출 헬퍼."""
    try:
        return _llm.chat.completions.create(
            model=_DEPLOY, messages=messages,
            temperature=0.0, max_tokens=max_tokens,
        ).choices[0].message.content
    except Exception:
        # reasoning 모델은 max_tokens 거부 → max_completion_tokens 재시도
        return _llm.chat.completions.create(
            model=_DEPLOY, messages=messages,
            max_completion_tokens=max_tokens,
        ).choices[0].message.content


def llm_draft_ground_truth(query: str, context: str) -> str:
    sys = ('당신은 RAG QA 정답 레이블러다. 주어진 컨텍스트만 근거로 정답을 생성한다. '
           '1~3문장으로 간결하게. 컨텍스트로 답할 수 없으면 정확히 "정보 부족" 만 출력하라.')
    usr = f'[질문]\n{query}\n\n[컨텍스트]\n{context}\n\n[정답을 한국어로 1~3문장으로 작성]'
    out = _llm_chat([{'role': 'system', 'content': sys},
                     {'role': 'user', 'content': usr}], max_tokens=200)
    return (out or '').strip()


def llm_judge_alignment(query: str, gt: str, response: str) -> int:
    """응답이 정답에 부합하는지 0(충돌)/1(부분)/2(완전)."""
    sys = '응답이 정답과 사실적으로 일치하는지 평가. 출력은 정확히 0,1,2 중 한 글자.'
    usr = (f'[질문] {query}\n[정답] {gt}\n[응답] {response}\n\n'
           '0=사실 충돌, 1=부분 일치, 2=완전 일치. 숫자만:')
    out = _llm_chat([{'role': 'system', 'content': sys},
                     {'role': 'user', 'content': usr}], max_tokens=4) or ''
    for ch in out:
        if ch in '012':
            return int(ch)
    return 1


def heuristic_draft(context: str) -> str:
    """LLM 미설정 시 fallback. 컨텍스트 핵심 라인 발췌형(약식). 검수 필수."""
    lines = [l.strip().lstrip('- ') for l in str(context).split('\n')
             if l.strip() and not l.strip().startswith('[')]
    info = [l for l in lines if any(c.isdigit() for c in l) or ':' in l]
    return '; '.join(info[:3]) if info else '정보 부족'


def label_one(row: pd.Series) -> pd.Series:
    q, c, r = row['query'], str(row['context']), str(row['response'])
    if USE_LLM_LABELER:
        gt = llm_draft_ground_truth(q, c)
        if not gt or gt == '정보 부족':
            return pd.Series({'ground_truth': '정보 부족', 'confidence': 'low',
                              'needs_review': True, 'judge_score': None})
        score = llm_judge_alignment(q, gt, r)
        confidence = {2: 'high', 1: 'medium', 0: 'low'}[score]
        return pd.Series({'ground_truth': gt, 'confidence': confidence,
                          'needs_review': confidence != 'high', 'judge_score': score})
    # fallback
    gt = heuristic_draft(c)
    return pd.Series({'ground_truth': gt,
                      'confidence': 'low',  # 휴리스틱은 항상 검수 대상
                      'needs_review': True, 'judge_score': None})


print(f'\n[A,B] {len(df_eval)}건 GT 초안 + judge 진행 중...')
records = []
for i, (_, row) in enumerate(df_eval.iterrows(), 1):
    records.append(label_one(row))
    if i % 25 == 0:
        print(f'  ... {i}/{len(df_eval)}')
gt_results = pd.DataFrame(records, index=df_eval.index)
df_eval = pd.concat(
    [df_eval.drop(columns=[c for c in ['ground_truth', 'confidence', 'needs_review', 'judge_score']
                           if c in df_eval.columns]),
     gt_results], axis=1)

# 자동 라벨 상태 구분: 이 단계에서는 아직 human-reviewed Gold가 아님
risk_order = {'hallucination': 0, 'rag_fail': 1, 'incomplete': 2, 'good': 3}
df_eval['_risk_priority'] = df_eval['weak_outcome'].map(risk_order).fillna(4).astype(int)
df_eval['label_status'] = np.where(
    df_eval['needs_review'],
    'review_queue_candidate',
    'silver_auto_labeled',
)
df_eval['is_human_reviewed'] = False

print('\n--- confidence 분포 ---')
print(df_eval['confidence'].value_counts().to_string())
print(f"검수 필요: {df_eval['needs_review'].sum()}건 "
      f"({df_eval['needs_review'].mean()*100:.1f}%)")
print('\n--- label_status 분포 ---')
print(df_eval['label_status'].value_counts().to_string())

# [C] 검수 우선순위 큐
priority = {'low': 0, 'medium': 1, 'high': 2}
df_eval['_review_priority'] = df_eval['confidence'].map(priority).fillna(3).astype(int)
review_queue = (
    df_eval[df_eval['needs_review']]
    .sort_values(['_review_priority', '_risk_priority', 'domain'])
    .head(20)
)
df_eval = df_eval.drop(columns=['_review_priority', '_risk_priority'])
print(f'\n[C] 검수 우선순위 큐 상위 {len(review_queue)}건 (저신뢰 + weak_outcome 위험 신호 우선)')

# 검수 큐 export (사람 검수 handoff)
review_queue_path = DATA_DIR / 'label_review_queue.csv'
review_cols = [
    'source_id', 'domain', 'weak_outcome', 'label_status',
    'query', 'response', 'ground_truth', 'confidence', 'needs_review',
]
review_queue[review_cols].to_csv(review_queue_path, index=False, encoding='utf-8-sig')
print(f"[C] 검수 큐 저장: {review_queue_path} ({len(review_queue)}건)")

# [E] Agent expected_tool_calls 후보 생성
# raw_logs 에서 weak_outcome=='good' 으로 추정되고 성공 tool_calls가 있는 케이스의 실제 호출을 후보로 채택
# 주의: 이 값도 자동 후보이며, 실무에서는 사람 검수 후 expected_tool_calls Gold로 승격해야 합니다.
weak_outcome_map = df_eval.set_index('source_id')['weak_outcome'].to_dict()
expected_tools_map = {}
for log in raw_logs:
    sid = f"{log.get('session_id')}_t{log.get('turn_index', 0)}"
    if weak_outcome_map.get(sid) != 'good':
        continue
    msg = str(log.get('assistant_message', ''))
    if log.get('error_flag') or msg.endswith('...'):
        continue
    if any(k in msg for k in HALLUC_KEYS) or msg.startswith('제공된 정보에서 관련 내용을 찾기 어렵습니다'):
        continue
    tcs = log.get('tool_calls') or []
    success_calls = [
        {'tool_name': t.get('tool_name'), 'tool_input': t.get('tool_input')}
        for t in tcs if t.get('status') != 'failed' and t.get('tool_name')
    ]
    if success_calls:
        expected_tools_map[sid] = success_calls

df_eval['expected_tool_calls'] = df_eval['source_id'].map(expected_tools_map)
print(f"[E] expected_tool_calls 후보가 있는 행: "
      f"{df_eval['expected_tool_calls'].notna().sum()}건 / {len(df_eval)}건")

# expected_tool_calls export (Agent 평가 정답 trajectory 검수/보강 handoff)
expected_calls_path = DATA_DIR / 'expected_tool_calls.jsonl'
expected_calls = df_eval[df_eval['expected_tool_calls'].notna()][
    ['source_id', 'domain', 'weak_outcome', 'label_status', 'query', 'expected_tool_calls']
].to_dict(orient='records')
with open(expected_calls_path, 'w', encoding='utf-8') as f:
    for rec in expected_calls:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')
print(f"[E] expected_tool_calls 후보 저장: {expected_calls_path} ({len(expected_calls)}건)")

# [D] Calibration / IAA 가이드 출력
print('''
[D] Calibration / Inter-Annotator Agreement 가이드
   사람이 label_review_queue.csv를 검수한 뒤 최소한 아래를 확인하세요.
   - judge-human agreement
   - false pass rate / false fail rate
   - threshold calibration
   - 2+ 어노테이터가 있으면 cohen_kappa_score 또는 Krippendorff alpha
   기준: κ ≥ 0.6 substantial, κ ≥ 0.8 almost perfect.
   κ가 낮으면 Rubric을 더 정교화한 뒤 재라운드.
''')

print('\n--- Ground Truth 초안 샘플 (high 신뢰) ---')
high = df_eval[df_eval['confidence'] == 'high']
if len(high) > 0:
    s = high.iloc[0]
    print(f'  Query        : {s["query"]}')
    print(f'  Weak_outcome : {s["weak_outcome"]}')
    print(f'  Response     : {str(s["response"])[:80]}...')
    print(f'  Ground Truth : {s["ground_truth"]}')
    print('  Status       : silver_auto_labeled (human review 전)')
else:
    print('  (high 신뢰 샘플 없음 — 검수 큐 확인 필요)')



[정답 레이블링 Rubric]
- 정답은 사용자 질문에 직접 답하는 1~3 문장이어야 한다 (컨텍스트 발췌가 아닌 답변).
- 컨텍스트에 명시된 사실에만 근거한다. 추론/외부지식 금지.
- 수치/정책 조건은 컨텍스트의 표현을 우선한다.
- 컨텍스트로 답할 수 없으면 정확히 "정보 부족" 으로 표기 (needs_review=True).
- 부분 정답 인정: 핵심 사실 누락 없으면 high, 일부 누락 medium, 사실 충돌 low.

LLM 레이블러 사용 가능: True

[A,B] 80건 GT 초안 + judge 진행 중...
  ... 25/80
  ... 50/80
  ... 75/80

--- confidence 분포 ---
confidence
low       55
medium    20
high       5
검수 필요: 75건 (93.8%)

--- label_status 분포 ---
label_status
review_queue_candidate    75
silver_auto_labeled        5

[C] 검수 우선순위 큐 상위 20건 (저신뢰 + weak_outcome 위험 신호 우선)
[C] 검수 큐 저장: data/label_review_queue.csv (20건)
[E] expected_tool_calls 후보가 있는 행: 17건 / 80건
[E] expected_tool_calls 후보 저장: data/expected_tool_calls.jsonl (17건)

[D] Calibration / Inter-Annotator Agreement 가이드
   사람이 label_review_queue.csv를 검수한 뒤 최소한 아래를 확인하세요.
   - judge-human agreement
   - false pass rate / false fail rate
   - threshold calibration
   - 2+ 어노테이터가 있으면 cohen_kappa_score 또는 Krippendorff alpha
   기준: κ ≥ 0

## 9단계: 데이터 분할 (Splitting)

평가 후보 데이터를 **평가 라이프사이클 관점에서** 3개 세트로 분할합니다.

### 평가 맥락에서의 의미 재해석

> ⚠️ 이 분할은 **모델 학습용**이 아니라 **평가 라이프사이클**용입니다. 이름은 ML 관습을 따르지만 의미가 다릅니다.

또 하나 중요한 점은, 이 노트북 단계에서의 Test/Holdout은 **human review 전에는 아직 Gold가 아니라 후보 세트**라는 것입니다.  
공식 배포 게이트나 회귀 비교에 쓰려면 8.5단계의 Review Queue를 거쳐 `is_human_reviewed=True` 상태로 승격된 데이터만 사용해야 합니다.

| 세트 | 비율 | 평가 맥락 의미 | 변경 가능 | Gold 조건 |
|------|------|------|-----------|-----------|
| **Development** | 60% | Evaluator 자체를 만들고 튜닝(프롬프트 변경, 임계값 조정 등)할 때 반복적으로 돌리는 셋 | ✅ 추가/수정 가능 | Silver도 사용 가능 |
| **Test** | 20% | 모델/에이전트 변경 후 공식 리포트용 점수를 산출하는 후보 셋 | ❌ 고정 | 사람 검수 후 Gold로 사용 |
| **Holdout** | 20% | 분기/반기마다 회귀 비교용으로 봉인하는 후보 셋 | ❌ 고정 | 사람 검수 후 Gold로 사용 |

### 분할 시 주의사항

1. **그룹 기반 분할**: 같은 `session_id`의 턴은 같은 세트로 배정 (데이터 누출 방지)
2. **도메인 균형**: 각 세트에 도메인이 균등하게 분포되도록 계층화
3. **Test/Holdout 고정**: 한번 분할된 Test/Holdout은 변경하지 않음
4. **Gold 승격 조건**: 공식 점수용 Test/Holdout은 human-reviewed 데이터만 사용
5. **샘플 수 확보**: 7.5단계에서 충분한 후보 표본을 뽑지 않으면 분할 후 셋별 표본이 작아져 통계 신뢰구간이 넓어짐


In [ ]:
def stratified_group_split(
    df: pd.DataFrame,
    group_col: str = 'session_id',
    stratify_col: str = 'domain',
    test_size: float = 0.2,
    holdout_size: float = 0.2,
    random_state: int = 42,
) -> tuple:
    """
    그룹 기반 계층화 분할
    - 같은 세션의 모든 턴은 같은 세트로 배정
    - 도메인 비율 유지
    """
    # 세션별 대표 도메인 결정 (최빈 도메인)
    session_domain = df.groupby(group_col)[stratify_col].agg(lambda x: x.mode().iloc[0])
    sessions = session_domain.reset_index()
    sessions.columns = [group_col, stratify_col]
    
    # 1차 분할: dev vs (test + holdout)
    combined_size = test_size + holdout_size
    dev_sessions, test_holdout_sessions = train_test_split(
        sessions, test_size=combined_size, stratify=sessions[stratify_col],
        random_state=random_state
    )
    
    # 2차 분할: test vs holdout
    holdout_ratio = holdout_size / combined_size
    test_sessions, holdout_sessions = train_test_split(
        test_holdout_sessions, test_size=holdout_ratio,
        stratify=test_holdout_sessions[stratify_col],
        random_state=random_state
    )
    
    # 원본 데이터에 매핑
    df_dev = df[df[group_col].isin(dev_sessions[group_col])].copy()
    df_test = df[df[group_col].isin(test_sessions[group_col])].copy()
    df_holdout = df[df[group_col].isin(holdout_sessions[group_col])].copy()
    
    return df_dev, df_test, df_holdout


df_dev, df_test, df_holdout = stratified_group_split(df_eval)

print('=' * 60)
print('📊 평가 후보 데이터 분할 결과')
print('=' * 60)
for name, df_split in [('Development', df_dev), ('Test candidate', df_test), ('Holdout candidate', df_holdout)]:
    print(f'\n{name}: {len(df_split)}건 ({len(df_split)/len(df_eval)*100:.1f}%)')
    print(f'  세션: {df_split["session_id"].nunique()}개')
    print(f'  도메인 분포: {df_split["domain"].value_counts().to_dict()}')
    if 'label_status' in df_split.columns:
        print(f'  라벨 상태: {df_split["label_status"].value_counts().to_dict()}')
    if 'is_human_reviewed' in df_split.columns:
        print(f'  human-reviewed: {int(df_split["is_human_reviewed"].sum())}건 / {len(df_split)}건')

# 데이터 누출 검증
dev_sessions = set(df_dev['session_id'])
test_sessions = set(df_test['session_id'])
holdout_sessions = set(df_holdout['session_id'])

assert dev_sessions.isdisjoint(test_sessions), '❌ Dev-Test 세션 중복!'
assert dev_sessions.isdisjoint(holdout_sessions), '❌ Dev-Holdout 세션 중복!'
assert test_sessions.isdisjoint(holdout_sessions), '❌ Test-Holdout 세션 중복!'
print(f'\n✅ 데이터 누출 검증 통과: 세트 간 세션 중복 없음')
print('⚠️ Test/Holdout은 human review 전까지 공식 Gold가 아니라 후보 세트입니다.')


📊 평가 후보 데이터 분할 결과

Development: 48건 (60.0%)
  세션: 45개
  도메인 분포: {'card': 16, 'account': 12, 'loan': 11, 'savings': 9}
  라벨 상태: {'review_queue_candidate': 45, 'silver_auto_labeled': 3}
  human-reviewed: 0건 / 48건

Test candidate: 16건 (20.0%)
  세션: 15개
  도메인 분포: {'card': 5, 'account': 5, 'loan': 4, 'savings': 2}
  라벨 상태: {'review_queue_candidate': 14, 'silver_auto_labeled': 2}
  human-reviewed: 0건 / 16건

Holdout candidate: 16건 (20.0%)
  세션: 16개
  도메인 분포: {'card': 5, 'account': 4, 'loan': 4, 'savings': 3}
  라벨 상태: {'review_queue_candidate': 16}
  human-reviewed: 0건 / 16건

✅ 데이터 누출 검증 통과: 세트 간 세션 중복 없음
⚠️ Test/Holdout은 human review 전까지 공식 Gold가 아니라 후보 세트입니다.


## 10단계: RAG 평가용 내보내기 (단일 턴 query/response/context)

다운스트림 노트북(01~05)이 사용하는 표준 파일명에 맞춰 내보냅니다.  
시나리오 무관 표준명을 유지하므로 다운스트림 코드 수정이 필요 없습니다.

| 파일 | 필드 | 포맷 | 사용처 |
|------|--------|------|--------|
| `data/sample_qa.json` | `question`, `agent`, `response`, `context`, `ground_truth` | JSON Array | 01_QA, 04, 05_Custom |
| `data/eval_input.jsonl` | `query`, `response`, `context` | JSONL | 02_Retrieval, 03_Groundedness |
| `data/sample_banking_qa.jsonl` | `query`, `response`, `context`, `is_correct` | JSONL | 07 |
| `data/eval_test.jsonl` / `data/eval_holdout.jsonl` | `query`, `response`, `context`, `ground_truth`, `domain`, `source_id` | JSONL | 보존 (회귀 평가용) |
| `data/label_review_queue.csv` | `source_id`, `query`, `response`, `ground_truth`, `confidence`, `needs_review` | CSV | 도메인 전문가 검수 작업 큐 |
| `data/expected_tool_calls.jsonl` | `source_id`, `query`, `expected_tool_calls` | JSONL | Agent 정답 trajectory 검수/보강 |


In [ ]:
def validate_schema(records: list, required_fields: list, name: str):
    """스키마 검증"""
    errors = []
    for i, record in enumerate(records):
        for field in required_fields:
            if field not in record or record[field] is None:
                errors.append(f'  Record {i}: missing or empty "{field}"')
    if errors:
        print(f'❌ {name} 검증 실패:')
        for e in errors[:5]:
            print(e)
        return False
    print(f'✅ {name} 검증 통과 ({len(records)}건, 필수 필드: {required_fields})')
    return True


# Development 세트로 주요 평가 데이터 생성 (Test/Holdout은 별도 저장)
# 실제 환경에서는 Test 세트로 최종 평가를 수행합니다.

# --- 1) sample_qa.json (question 필드, JSON Array) ---
qa_records = []
for _, row in df_dev.iterrows():
    qa_records.append({
        'question': row['query'],
        'agent': f"{row['domain']}-agent",
        'response': row['response'],
        'context': row['context'],
        'ground_truth': row['ground_truth'],
    })

validate_schema(qa_records, ['question', 'response', 'context'], 'sample_qa.json')
with open(DATA_DIR / 'sample_qa.json', 'w', encoding='utf-8') as f:
    json.dump(qa_records, f, ensure_ascii=False, indent=2)


# --- 2) eval_input.jsonl (query 필드, JSONL) ---
eval_records = []
for _, row in df_dev.iterrows():
    eval_records.append({
        'query': row['query'],
        'response': row['response'],
        'context': row['context'],
    })

validate_schema(eval_records, ['query', 'response', 'context'], 'eval_input.jsonl')
with open(DATA_DIR / 'eval_input.jsonl', 'w', encoding='utf-8') as f:
    for r in eval_records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')


# --- 3) sample_banking_qa.jsonl (query + is_correct, JSONL) ---
# 다운스트림 07 노트북에서 직접 사용하는 파일.
banking_records = []
for _, row in df_dev.iterrows():
    banking_records.append({
        'query': row['query'],
        'response': row['response'],
        'context': row['context'],
        'is_correct': row['confidence'] == 'high',
    })

validate_schema(banking_records, ['query', 'response', 'context', 'is_correct'], 'sample_banking_qa.jsonl')
with open(DATA_DIR / 'sample_banking_qa.jsonl', 'w', encoding='utf-8') as f:
    for r in banking_records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')


# --- 4) Test/Holdout 세트 별도 저장 ---
for name, df_split in [('test', df_test), ('holdout', df_holdout)]:
    records = []
    for _, row in df_split.iterrows():
        records.append({
            'query': row['query'],
            'response': row['response'],
            'context': row['context'],
            'ground_truth': row['ground_truth'],
            'domain': row['domain'],
            'source_id': row['source_id'],
        })
    path = DATA_DIR / f'eval_{name}.jsonl'
    with open(path, 'w', encoding='utf-8') as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(f'✓ {path} 저장 ({len(records)}건)')


print(f'\n{"=" * 60}')
print(f'📁 RAG 평가용 내보내기 완료  (banking)')
print(f'{"=" * 60}')
for f in sorted(DATA_DIR.glob('*')):
    print(f'  {f.name:40s} {f.stat().st_size:>8,} bytes')


✅ sample_qa.json 검증 통과 (48건, 필수 필드: ['question', 'response', 'context'])
✅ eval_input.jsonl 검증 통과 (48건, 필수 필드: ['query', 'response', 'context'])
✅ sample_banking_qa.jsonl 검증 통과 (48건, 필수 필드: ['query', 'response', 'context', 'is_correct'])
✓ data/eval_test.jsonl 저장 (16건)
✓ data/eval_holdout.jsonl 저장 (16건)

📁 RAG 평가용 내보내기 완료  (banking)
  agent_eval_traces.jsonl                   968,816 bytes
  eval_holdout.jsonl                         16,147 bytes
  eval_input.jsonl                           29,803 bytes
  eval_test.jsonl                            15,984 bytes
  expected_tool_calls.jsonl                   6,297 bytes
  label_review_queue.csv                      8,065 bytes
  multi_turn_conversations_samples.json      14,254 bytes
  raw_chat_logs.jsonl                       845,969 bytes
  sample_banking_qa.jsonl                    30,808 bytes
  sample_qa.json                             38,063 bytes


## 11단계: Agent Trace 형식 변환 (06, 08 노트북 연결)

지금까지 만든 파일들은 **단일 턴 RAG 평가**(`query / response / context`)용 입니다.  
이 노트북의 핵심 메시지인 **"운영 중인 AI Agent를 평가한다"** 를 완성하려면,  
운영 trace 로그를 **Agent 평가 SDK가 요구하는 OpenAI 메시지 스키마**로 변환해야 합니다.

### 왜 이 단계가 필요한가

| 평가 대상 | 필요 데이터 형식 | 사용 노트북 |
|----------|------------------|-------------|
| RAG 챗봇 (단일 턴) | `query, response, context` (string) | 01, 02, 03, 04, 05 |
| **Tool-calling Agent** | `query[], response[], tool_definitions[]` (messages array) | **06, 08** |

### `scenario` vs `scenario_label` (용어 구분)

| 필드 | 의미 | 예시 |
|---|---|---|
| `scenario` | 데이터셋의 비즈니스 도메인/실험 묶음 | `banking` |
| `scenario_label` | 개별 trace 턴의 품질/행동 라벨(자동 추론) | `correct_tool_use`, `tool_call_failure`, `hallucination_no_tool` |

> 즉, `scenario`는 **데이터가 어디서 왔는지**, `scenario_label`은 **해당 턴에서 무슨 품질 이벤트가 있었는지**를 의미합니다.

### 변환 매핑

```
raw_log                           →  agent_trace
──────────────────────────────────────────────────────────────
domain                            →  domain
session_id, turn_index            →  session_id, turn_index
user_message                      →  query[1].content (role=user)
domain → DOMAIN_SYSTEM_PROMPTS    →  query[0].content (role=system)
tool_calls[i].tool_name/input     →  response[2i].content (role=assistant, type=tool_call)
tool_calls[i].tool_output         →  response[2i+1].content (role=tool, type=tool_result)
assistant_message                 →  response[-1].content (role=assistant, 최종 텍스트)
TOOL_CALL_TEMPLATES[domain]       →  tool_definitions[] (OpenAI function-calling schema)
                                  +  scenario_label (자동 추론: hallucination / multi_step / tool_call_failure …)
```

### 출력 파일

| 파일 | 용도 |
|------|------|
| `data/agent_eval_traces.jsonl` | 06, 08 입력 (선택된 시나리오의 trace만 포함) |

> 💡 정제 단계(`df_clean`)가 아닌 **PII 제거 직후의 `df_raw`** 를 사용합니다.  
> Agent 평가는 **할루시네이션·잘못된 도구 선택·도구 호출 실패** 같은 케이스도 평가 대상이므로 살려둬야 합니다.


In [14]:
# ============================================================
# 운영 로그 → Agent Trace 변환 (06, 08 노트북용)
# ============================================================

def _safe_list(value) -> list:
    """pandas dict 변환 시 NaN(float)으로 바뀐 list 필드를 안전하게 처리"""
    return value if isinstance(value, list) else []


def _infer_param_type(value) -> str:
    """파라미터 값에서 OpenAI function-calling schema 타입 추론"""
    if isinstance(value, bool):  return 'boolean'
    if isinstance(value, int):   return 'integer'
    if isinstance(value, float): return 'number'
    if isinstance(value, dict):  return 'object'
    if isinstance(value, list):  return 'array'
    return 'string'


def build_tool_definitions(domain: str) -> list:
    """도메인의 TOOL_CALL_TEMPLATES → OpenAI function-calling schema"""
    tools = TOOL_CALL_TEMPLATES.get(domain, [])
    definitions = []
    for tool in tools:
        properties = {}
        required = []
        for key, val in tool.get('tool_input', {}).items():
            properties[key] = {'type': _infer_param_type(val)}
            required.append(key)
        definitions.append({
            'type': 'function',
            'name': tool['tool_name'],
            'description': f"{tool['tool_name']} (auto-generated from operation log template)",
            'parameters': {
                'type': 'object',
                'properties': properties,
                'required': required,
            },
        })
    return definitions


def classify_scenario_label(record: dict) -> str:
    """raw_log 한 건을 보고 평가 분석용 라벨 추론.

    출력 라벨: correct_tool_use / multi_step_correct / no_tool_used /
              tool_call_failure / hallucination_no_tool / incomplete_response
    """
    tool_calls = _safe_list(record.get('tool_calls'))
    has_tools = bool(tool_calls)
    has_failed_tool = any(t.get('status') == 'failed' for t in tool_calls)
    msg = str(record.get('assistant_message', ''))

    halluc_keywords = ('수수료 0원', '특별 지원금', '한시적', '무재해 365일', '항상 99%', '0건', '영구적')
    if not has_tools and any(k in msg for k in halluc_keywords):
        return 'hallucination_no_tool'
    if has_failed_tool:
        return 'tool_call_failure'
    if not has_tools:
        return 'no_tool_used'
    if msg.endswith('...'):
        return 'incomplete_response'
    if len(tool_calls) > 1:
        return 'multi_step_correct'
    return 'correct_tool_use'


def convert_log_to_agent_trace(record: dict) -> dict:
    """raw_log 한 건 → Azure AI Evaluation SDK Agent 평가 형식

    출력 스키마: {scenario, domain, scenario_label, session_id, turn_index,
                   query[], response[], tool_definitions[]}
    """
    domain = record.get('domain', 'unknown')
    system_prompt = DOMAIN_SYSTEM_PROMPTS.get(
        domain, '당신은 운영 에이전트입니다. 제공된 도구를 사용하여 정확하게 답변하세요.'
    )

    query = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': str(record.get('user_message', ''))},
    ]

    response = []
    sess = record.get('session_id', 'sess')
    turn = record.get('turn_index', 0)
    for idx, tc in enumerate(_safe_list(record.get('tool_calls'))):
        call_id = tc.get('call_id') or f'call_{sess}_{turn}_{idx}'
        response.append({
            'role': 'assistant',
            'content': [{
                'type': 'tool_call',
                'tool_call_id': call_id,
                'name': tc.get('tool_name', ''),
                'arguments': tc.get('tool_input', {}),
            }],
        })
        response.append({
            'role': 'tool',
            'tool_call_id': call_id,
            'content': [{
                'type': 'tool_result',
                'tool_call_id': call_id,
                'tool_result': tc.get('tool_output', ''),
            }],
        })
    final_msg = record.get('assistant_message')
    if final_msg is not None and not (isinstance(final_msg, float) and pd.isna(final_msg)):
        response.append({'role': 'assistant', 'content': str(final_msg)})

    turn_idx = record.get('turn_index', 0)
    if isinstance(turn_idx, float) and pd.isna(turn_idx):
        turn_idx = 0

    return {
        'scenario': 'banking',
        'domain': domain,
        'scenario_label': classify_scenario_label(record),
        'session_id': record.get('session_id'),
        'turn_index': int(turn_idx),
        'query': query,
        'response': response,
        'tool_definitions': build_tool_definitions(domain),
    }


# ------------------------------------------------------------
# 변환 대상: PII 제거 직후의 df_raw 사용
# (정제된 df_clean이 아님 — 할루시네이션/wrong_tool/incomplete도 평가 대상이므로 살려둠)
# 단, 응답이 완전히 비어있는 error 케이스만 제외
# ------------------------------------------------------------
if 'error_flag' in df_raw.columns:
    err_col = df_raw['error_flag'].fillna(False).astype(bool)
else:
    err_col = pd.Series(False, index=df_raw.index)

trace_source_mask = (
    (~err_col) &
    (df_raw['assistant_message'].notna()) &
    (df_raw['assistant_message'].astype(str).str.len() > 0) &
    (df_raw['assistant_message'].astype(str) != 'None')
)
trace_source = df_raw[trace_source_mask].to_dict(orient='records')

agent_traces = [convert_log_to_agent_trace(r) for r in trace_source]

trace_path = DATA_DIR / 'agent_eval_traces.jsonl'
with open(trace_path, 'w', encoding='utf-8') as f:
    for t in agent_traces:
        f.write(json.dumps(t, ensure_ascii=False) + '\n')

# ------------------------------------------------------------
# 요약
# ------------------------------------------------------------
print('=' * 60)
print('🔗 운영 로그 → Agent Trace 변환 완료  (banking)')
print('=' * 60)
print(f'원본 raw 로그:    {len(df_raw)}건')
print(f'변환된 trace:     {len(agent_traces)}건')
print(f'저장 파일:        {trace_path}')

print('\n--- 도메인 분포 ---')
for dom, cnt in Counter(t['domain'] for t in agent_traces).most_common():
    print(f'  {dom:20s} {cnt}건')

print('\n--- 평가 분석 라벨 분포 ---')
for label, cnt in Counter(t['scenario_label'] for t in agent_traces).most_common():
    print(f'  {label:25s} {cnt}건')

print('\n--- 변환 샘플 (1건, 일부) ---')
sample = agent_traces[0]
print(f"scenario:         {sample['scenario']}")
print(f"domain:           {sample['domain']}")
print(f"scenario_label:   {sample['scenario_label']}")
print(f"query roles:      {[m['role'] for m in sample['query']]}")
print(f"response roles:   {[m['role'] for m in sample['response']]}")
print(f"tool_definitions: {[td['name'] for td in sample['tool_definitions']]}")
print('\n→ 06_agent_evaluation.ipynb / 08_Agent_Evaluation.ipynb 의 입력으로 사용됩니다.')


🔗 운영 로그 → Agent Trace 변환 완료  (banking)
원본 raw 로그:    520건
변환된 trace:     487건
저장 파일:        data/agent_eval_traces.jsonl

--- 도메인 분포 ---
  card                 138건
  loan                 123건
  account              123건
  savings              103건

--- 평가 분석 라벨 분포 ---
  multi_step_correct        196건
  correct_tool_use          101건
  no_tool_used              76건
  incomplete_response       65건
  tool_call_failure         30건
  hallucination_no_tool     19건

--- 변환 샘플 (1건, 일부) ---
scenario:         banking
domain:           savings
scenario_label:   tool_call_failure
query roles:      ['system', 'user']
response roles:   ['assistant', 'tool', 'assistant']
tool_definitions: ['search_knowledge_base', 'get_product_detail', 'calculate_interest']

→ 06_agent_evaluation.ipynb / 08_Agent_Evaluation.ipynb 의 입력으로 사용됩니다.


/var/folders/0r/wx9n6dq14758w8v_tc4rdpbc0000gn/T/ipykernel_41837/179150220.py:133: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  err_col = df_raw['error_flag'].fillna(False).astype(bool)


## 12단계: 증분 업데이트 시나리오

운영 환경에서는 **새로운 trace 로그가 계속 쌓입니다**.  
기존 평가 데이터셋에 새 데이터를 안전하게 추가하는 방법을 보여줍니다.

### 증분 업데이트 원칙

| 원칙 | 설명 |
|------|------|
| **Test/Holdout 고정** | 기존 Test/Holdout 세트는 절대 변경하지 않음 (일관된 비교 기준 유지) |
| **Dev에만 추가** | 새 로그는 정제 후 Development 세트에만 추가 |
| **버전 관리** | 데이터셋 버전을 명시적으로 관리 |
| **품질 모니터링** | 새 데이터의 품질 분포가 기존과 크게 다르지 않은지 확인 |


In [15]:
def incremental_update(
    new_logs: list,
    existing_dev_path: Path,
    test_path: Path,
    holdout_path: Path,
    version: str = 'v2',
) -> pd.DataFrame:
    """
    증분 업데이트 파이프라인
    
    1. 새 로그에 동일한 전처리 파이프라인 적용
    2. Test/Holdout과의 중복 제거
    3. 기존 Dev 세트에 추가
    """
    # 1. 새 로그 전처리
    df_new = pd.DataFrame(new_logs)
    df_new['user_message'] = df_new['user_message'].apply(remove_pii)
    df_new['assistant_message'] = df_new['assistant_message'].apply(remove_pii)
    
    # 오류/빈 응답 제거
    df_new = df_new[
        (df_new['assistant_message'].notna()) &
        (df_new['assistant_message'].astype(str).str.len() >= 10) &
        (df_new['retrieved_context'].notna())
    ]
    
    # 2. Test/Holdout 세트의 query 로드 (중복 방지)
    protected_queries = set()
    for path in [test_path, holdout_path]:
        if path.exists():
            with open(path, 'r', encoding='utf-8') as f:
                for line in f:
                    record = json.loads(line)
                    protected_queries.add(record.get('query', ''))
    
    # 보호 세트와 중복되는 질문 제거
    before = len(df_new)
    df_new = df_new[~df_new['user_message'].isin(protected_queries)]
    print(f'  Test/Holdout 중복 제거: {before - len(df_new)}건')
    
    # 3. 스키마 변환
    df_new['source_id'] = df_new['session_id'] + '_t' + df_new.get('turn_index', 0).astype(str)
    df_new_eval = pd.DataFrame({
        'query': df_new['user_message'],
        'response': df_new['assistant_message'],
        'context': df_new['retrieved_context'],
        'domain': df_new['domain'],
        'source_id': df_new['source_id'],
        'version': version,
    })
    
    return df_new_eval


# 시뮬레이션: 50건의 새 로그 수신
new_raw_logs = generate_raw_logs(50)

print('=' * 60)
print('🔄 증분 업데이트 시뮬레이션')
print('=' * 60)
print(f'새 로그: {len(new_raw_logs)}건')

df_new_eval = incremental_update(
    new_raw_logs,
    existing_dev_path=DATA_DIR / 'eval_input.jsonl',
    test_path=DATA_DIR / 'eval_test.jsonl',
    holdout_path=DATA_DIR / 'eval_holdout.jsonl',
    version='v2',
)

print(f'\n정제 후 추가 가능 데이터: {len(df_new_eval)}건')
print(f'기존 Dev 세트: {len(df_dev)}건')
print(f'업데이트 후 예상 Dev 세트: {len(df_dev) + len(df_new_eval)}건')

print(f'\n--- 도메인 분포 (신규) ---')
if len(df_new_eval) > 0:
    print(df_new_eval['domain'].value_counts().to_string())

🔄 증분 업데이트 시뮬레이션
새 로그: 70건
  Test/Holdout 중복 제거: 26건

정제 후 추가 가능 데이터: 39건
기존 Dev 세트: 48건
업데이트 후 예상 Dev 세트: 87건

--- 도메인 분포 (신규) ---
domain
card       13
loan       12
savings    12
account     2


## 📋 요약 및 다음 단계

### 이 노트북에서 수행한 작업 (도메인: 은행)

| 단계 | 내용 | 결과 |
|------|------|------|
| 3 | Agent Trace 확보 (BYO 또는 시뮬레이션) | `data/raw_chat_logs.jsonl` |
| 4 | PII 제거 / 비식별화 | 개인정보 마스킹 완료 |
| 5 | EDA | 품질 분포 / 결측 / 도메인 분석 |
| 6 | 데이터 정제 | 오류·결측·근접 중복 제거 |
| 7 | 턴 단위 변환 + weak_outcome 추정 | `df_turns` (샘플링용 약식 위험 라벨 포함) |
| **7.5** | **⭐ 평가 후보 추출** (Cochran 후보 크기 + domain×weak_outcome quota + 다양성 + 위험 케이스 oversample) | `df_sampled` (평가 후보셋) |
| 8 | 평가 후보 스키마 변환 | `df_eval` |
| **8.5** | **⭐ GT 초안/검수 큐** (Rubric + LLM-assisted 초안 + judge confidence + Review Queue + expected_tool_calls 후보 + Calibration 가이드) | `ground_truth`, `confidence`, `needs_review`, `label_status`, `expected_tool_calls` |
| 9 | 후보 분할 (Dev/Test/Holdout, 평가 맥락) | 그룹 기반 계층화, 누출 검증, Gold 승격 전 후보 세트 |
| 10 | RAG 평가용 내보내기 | `sample_qa.json`, `eval_input.jsonl`, `sample_banking_qa.jsonl`, `eval_test.jsonl`, `eval_holdout.jsonl`, `label_review_queue.csv`, `expected_tool_calls.jsonl` |
| 11 | Agent Trace 형식 변환 | `agent_eval_traces.jsonl` |
| 12 | 증분 업데이트 시나리오 | Dev에만 추가, Test/Holdout 고정 |

### 데이터 → 평가 노트북 매핑

```
운영 trace 로그 (raw_chat_logs.jsonl)
    │
    ├─ 7 weak_outcome 추정 → 7.5 후보 추출 → 8 스키마 변환 → 8.5 GT 초안/검수 큐
    │       │
    │       ├─ map_to_eval_schema → sample_qa.json / eval_input.jsonl / sample_banking_qa.jsonl
    │       │                          ↓
    │       │                     01 (QA), 02 (Retrieval), 03/04 (Groundedness), 05 (Custom), 07 (Portal)
    │       │                     = RAG 챗봇 품질 평가 후보
    │       │
    │       └─ expected_tool_calls 후보 (Agent 정답 trajectory 검수 대상)
    │
    └─ convert_log_to_agent_trace → agent_eval_traces.jsonl
                                       ↓
                                 06 (Agent 기본), 08 (Agent 종합)
                                 = Tool-calling Agent 평가
```

### 중요한 해석 기준

- `weak_outcome`은 공식 정답 라벨이 아니라 샘플링과 검수 우선순위를 위한 약식 신호입니다.
- `Silver Set`은 자동 라벨 기반 탐색/튜닝용이며, 공식 KPI처럼 해석하면 안 됩니다.
- `Gold Set`은 사람 검수 완료 후에만 공식 점수, 배포 게이트, 회귀 비교에 사용할 수 있습니다.
- Test/Holdout은 분할 직후에는 후보 세트이며, human review를 거쳐 Gold로 승격되어야 합니다.

### 다음 단계

- 본 노트북 실행 완료 후 → `01_QA_Evaluator.ipynb` 부터 순차 진행
- 자기 trace를 가져왔다면 → 7.5 의 후보 표본 크기/quota 가중치를 조직 데이터에 맞게 조정
- 8.5 의 `label_review_queue.csv`를 도메인 전문가가 검수한 뒤 Gold/Test/Holdout으로 승격
